# Historical phi collection and analysis

Preserved code order from the submitted working notebook. Outputs and transient metadata were removed. This is an execution history, NOT a fresh-runtime Run all entry point: later cells depend on earlier runtime state and restored environments. Use the numbered analysis notebooks for report analyses. GPU collection can be expensive; run selected collection sections deliberately.

### Original cell 0
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import json
import os
import shutil

drive.mount("/content/drive")

PHI_MODEL = "microsoft/Phi-4-mini-instruct"

PHI_ROOT = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
)
PHI_ROOT.mkdir(parents=True, exist_ok=True)

OLD_DATA = Path(
    "/content/drive/MyDrive/B3IT_reproduction/data"
)

PHI_WORK = Path("/content/b3it_phi4mini")
PHI_WORK.mkdir(parents=True, exist_ok=True)

print("Phi results:", PHI_ROOT)

### Original cell 1
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
subprocess.run(["nvidia-smi"], check=True)

### Original cell 2
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
PHI_REPO = Path("/content/b3it_phi4mini_repo")
COMMIT = "ad2f69295797a5f03e7b480246080d91087d9007"

if not PHI_REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/timothee-chauvin/"
            "token-efficient-change-detection-llm-apis.git",
            str(PHI_REPO),
        ],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(PHI_REPO), "checkout", "--detach", COMMIT],
    check=True,
)

PHI_PROJECT = PHI_REPO / "track-llm-apis"
assert PHI_PROJECT.exists()

(PHI_ROOT / "repository_commit.txt").write_text(COMMIT)
print("Repository ready.")

### Original cell 3
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
subprocess.run(
    ["pip", "install", "-q", "uv"],
    check=True,
)

UV = shutil.which("uv")
assert UV

subprocess.run(
    [
        UV, "sync",
        "--project", str(PHI_PROJECT),
        "--python", "3.12",
        "--frozen",
        "--no-dev",
    ],
    check=True,
)

PHI_PYTHON = PHI_PROJECT / ".venv/bin/python"

subprocess.run(
    [
        UV, "pip", "install",
        "--python", str(PHI_PYTHON),
        "ninja",
    ],
    check=True,
)

print("Environment installed.")

### Original cell 4
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
PHI_ENV = os.environ.copy()
PHI_ENV.update({
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "TRACKLLM__DATA_DIR": str(OLD_DATA),
})

PHI_ENV["PATH"] = (
    str(PHI_PYTHON.parent)
    + os.pathsep
    + PHI_ENV.get("PATH", "")
)

def run_phi_script(script_path, log_name):
    log_path = PHI_ROOT / log_name

    print("Script:", script_path)
    print("Log:", log_path)

    with log_path.open("w") as log:
        process = subprocess.Popen(
            [str(PHI_PYTHON), str(script_path)],
            cwd=PHI_PROJECT,
            env=PHI_ENV,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()

            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise

    if returncode != 0:
        raise RuntimeError(
            f"Script failed ({returncode}). See {log_path}"
        )

### Original cell 5
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
prepare_code = r'''
import json
import math
from pathlib import Path

from transformers import AutoTokenizer, AutoConfig
from track_llm_apis.config import config
from track_llm_apis.util import load_lmsys_chat_1m


def main():
    model_name = "microsoft/Phi-4-mini-instruct"
    output_dir = Path(
        "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_config = AutoConfig.from_pretrained(model_name)

    dataset = load_lmsys_chat_1m(
        use_cache=True,
        datasets_dir=config.datasets_dir,
    )

    traffic = [
        row["conversation"][0]["content"]
        for row in dataset
    ]

    # Match raw-text inference: no added chat template.
    encoded = tokenizer(
        traffic,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_attention_mask=False,
    )

    lengths = [len(ids) for ids in encoded["input_ids"]]
    longest = max(lengths)

    # Leave room for the generated token and a small margin.
    serving_limit = max(
        2048,
        math.ceil((longest + 32) / 256) * 256,
    )

    if serving_limit > model_config.max_position_embeddings:
        raise RuntimeError(
            "A traffic prompt exceeds the model context limit."
        )

    report = {
        "model": model_name,
        "traffic_prompts": len(traffic),
        "maximum_traffic_tokens": longest,
        "max_model_len": serving_limit,
        "dtype": "float16",
        "enforce_eager": True,
        "gpu_memory_utilization": 0.85,
        "max_num_seqs": 64,
        "max_num_batched_tokens": 2048,
    }

    (output_dir / "inference_settings.json").write_text(
        json.dumps(report, indent=2)
    )

    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()
'''

prepare_path = PHI_WORK / "prepare.py"
prepare_path.write_text(prepare_code)
(PHI_ROOT / "prepare.py").write_text(prepare_code)

run_phi_script(prepare_path, "prepare.log")

### Original cell 6
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
test_code = r'''
import json
import random
from pathlib import Path

import numpy as np
import torch
from vllm import LLM

from track_llm_apis.config import config
from track_llm_apis.util import load_lmsys_chat_1m
from track_llm_apis.sampling.vllm_sampling import (
    vllm_inference_bi,
)


def main():
    root = Path(
        "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
    )
    settings = json.loads(
        (root / "inference_settings.json").read_text()
    )

    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)

    dataset = load_lmsys_chat_1m(
        use_cache=True,
        datasets_dir=config.datasets_dir,
    )
    traffic = [
        row["conversation"][0]["content"]
        for row in dataset
    ]

    llm = LLM(
        model=settings["model"],
        dtype=settings["dtype"],
        enforce_eager=settings["enforce_eager"],
        gpu_memory_utilization=settings["gpu_memory_utilization"],
        max_model_len=settings["max_model_len"],
        max_num_seqs=settings["max_num_seqs"],
        max_num_batched_tokens=settings["max_num_batched_tokens"],
        enable_chunked_prefill=True,
    )

    # Simple loading/inference probes, not identified Border Inputs.
    probes = ["x", "a", "the", "hello"]

    result = vllm_inference_bi(
        llm=llm,
        inputs=probes,
        traffic_prompts=traffic,
        n_samples=3,
        temperature=0.0,
        input_fraction=0.95,
        batch_size=64,
        max_tokens=1,
    )

    payload = {
        "settings": settings,
        "counts": result.counts,
        "prompt_lengths": result.prompt_lengths,
    }

    (root / "loading_test.json").write_text(
        json.dumps(payload, indent=2)
    )

    print("\nINFERENCE TEST COMPLETED")
    for prompt, counts in result.counts.items():
        print(repr(prompt), "->", counts)


if __name__ == "__main__":
    main()
'''

test_path = PHI_WORK / "loading_test.py"
test_path.write_text(test_code)
(PHI_ROOT / "loading_test.py").write_text(test_code)

print("Inference test script created.")

### Original cell 7
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
run_phi_script(
    test_path,
    "loading_test.log",
)

### Original cell 8
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
test_result = json.loads(
    (PHI_ROOT / "loading_test.json").read_text()
)

assert len(test_result["counts"]) == 4
assert all(
    sum(counts.values()) == 3
    for counts in test_result["counts"].values()
)

print("Phi-4-mini loaded and generated successfully.")

for prompt, counts in test_result["counts"].items():
    print(repr(prompt), "->", counts)

subprocess.run(["nvidia-smi"], check=True)

### Original cell 9
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
candidate_code = r'''
import json
import random
from pathlib import Path

import numpy as np
from transformers import AutoTokenizer

from track_llm_apis.sampling.bi_phase_1 import (
    get_single_token_inputs,
)


def main():
    root = Path(
        "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
    )

    random.seed(0)
    np.random.seed(0)

    tokenizer = AutoTokenizer.from_pretrained(
        "microsoft/Phi-4-mini-instruct"
    )

    candidates = get_single_token_inputs(tokenizer, 20000)

    assert isinstance(candidates, list)
    assert all(isinstance(prompt, str) for prompt in candidates)

    payload = {
        "model": "microsoft/Phi-4-mini-instruct",
        "seed": 0,
        "candidates": candidates,
    }

    path = root / "candidate_prompts.json"

    if path.exists():
        assert json.loads(path.read_text()) == payload, (
            "Existing candidates differ."
        )
    else:
        path.write_text(json.dumps(payload))

    print("Candidate count:", len(candidates))
    print("Examples:", [repr(p) for p in candidates[:10]])


if __name__ == "__main__":
    main()
'''

candidate_path = PHI_WORK / "prepare_candidates.py"
candidate_path.write_text(candidate_code)
(PHI_ROOT / "prepare_candidates.py").write_text(candidate_code)

run_phi_script(
    candidate_path,
    "prepare_candidates.log",
)

### Original cell 10
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
discovery_code = r'''
import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch
from vllm import LLM

from track_llm_apis.config import config
from track_llm_apis.util import load_lmsys_chat_1m
from track_llm_apis.sampling.vllm_sampling import (
    vllm_inference_bi,
)


def atomic_json(path, payload):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload))
    temporary.replace(path)


def main():
    root = Path(
        "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
    )
    output_dir = root / "discovery_v1"
    output_dir.mkdir(exist_ok=True)

    settings = json.loads(
        (root / "inference_settings.json").read_text()
    )
    candidate_bytes = (root / "candidate_prompts.json").read_bytes()
    candidates = json.loads(candidate_bytes)["candidates"]

    chunk_size = 1000

    manifest = {
        "model": settings["model"],
        "inference_settings": settings,
        "candidate_sha256": hashlib.sha256(
            candidate_bytes
        ).hexdigest(),
        "chunk_size": chunk_size,
        "samples_per_candidate": 3,
        "temperature": 0.0,
        "batch_size": 64,
        "input_fraction": 0.95,
        "seed_rule": "chunk index, starting at zero",
    }

    manifest_path = output_dir / "manifest.json"
    if manifest_path.exists():
        assert json.loads(manifest_path.read_text()) == manifest
    else:
        atomic_json(manifest_path, manifest)

    chunks = [
        candidates[start:start + chunk_size]
        for start in range(0, len(candidates), chunk_size)
    ]

    pending = []

    for index, prompts in enumerate(chunks):
        path = output_dir / f"chunk_{index:03d}.json"

        if path.exists():
            saved = json.loads(path.read_text())
            assert saved["prompts"] == prompts
            assert all(
                sum(saved["counts"][p].values()) == 3
                for p in prompts
            )
        else:
            pending.append((index, prompts, path))

    if not pending:
        print("All discovery chunks already completed.")
        return

    dataset = load_lmsys_chat_1m(
        use_cache=True,
        datasets_dir=config.datasets_dir,
    )
    traffic = [
        row["conversation"][0]["content"]
        for row in dataset
    ]

    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)

    llm = LLM(
        model=settings["model"],
        dtype=settings["dtype"],
        enforce_eager=settings["enforce_eager"],
        gpu_memory_utilization=settings["gpu_memory_utilization"],
        max_model_len=settings["max_model_len"],
        max_num_seqs=settings["max_num_seqs"],
        max_num_batched_tokens=settings["max_num_batched_tokens"],
        enable_chunked_prefill=True,
    )

    for index, prompts, path in pending:
        random.seed(index)
        np.random.seed(index)
        torch.manual_seed(index)

        print(
            f"\nCHUNK {index + 1}/{len(chunks)}: "
            f"{len(prompts)} candidates",
            flush=True,
        )

        result = vllm_inference_bi(
            llm=llm,
            inputs=prompts,
            traffic_prompts=traffic,
            n_samples=3,
            temperature=0.0,
            input_fraction=0.95,
            batch_size=64,
            max_tokens=1,
        )

        atomic_json(path, {
            "prompts": prompts,
            "counts": result.counts,
            "prompt_lengths": result.prompt_lengths,
        })

        n_bis = sum(
            len(counts) >= 2
            for counts in result.counts.values()
        )
        print(
            f"Saved chunk {index + 1}; "
            f"Border Inputs found: {n_bis}",
            flush=True,
        )

    print("\nDISCOVERY COMPLETED", flush=True)


if __name__ == "__main__":
    main()
'''

discovery_path = PHI_WORK / "discover.py"
discovery_path.write_text(discovery_code)
(PHI_ROOT / "discover.py").write_text(discovery_code)

import ast
ast.parse(discovery_code)

print("Discovery script created; syntax checked.")

### Original cell 11
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
run_phi_script(
    discovery_path,
    "discovery.log",
)

### Original cell 12
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
candidate_data = json.loads(
    (PHI_ROOT / "candidate_prompts.json").read_text()
)
candidates = candidate_data["candidates"]

discovery_dir = PHI_ROOT / "discovery_v1"

all_counts = {}
all_lengths = {}

for path in sorted(discovery_dir.glob("chunk_*.json")):
    chunk = json.loads(path.read_text())

    for prompt, counts in chunk["counts"].items():
        assert prompt not in all_counts, (
            "Duplicate candidate across chunks."
        )
        all_counts[prompt] = counts

    all_lengths.update(chunk["prompt_lengths"])

assert len(all_counts) == len(candidates), (
    "Discovery is incomplete. Rerun Cell 12."
)

phi_border_inputs = [
    prompt
    for prompt in candidates
    if len(all_counts[prompt]) >= 2
]

payload = {
    "model": PHI_MODEL,
    "tested_candidates": len(all_counts),
    "border_inputs": phi_border_inputs,
    "counts": {
        prompt: all_counts[prompt]
        for prompt in phi_border_inputs
    },
    "prompt_lengths": {
        prompt: all_lengths[prompt]
        for prompt in phi_border_inputs
    },
}

(PHI_ROOT / "discovered_border_inputs.json").write_text(
    json.dumps(payload, indent=2)
)

print("Candidates tested:", len(all_counts))
print("Border Inputs discovered:", len(phi_border_inputs))

for prompt in phi_border_inputs[:10]:
    print(repr(prompt), "->", all_counts[prompt])

if len(phi_border_inputs) < 10:
    print(
        "\nFewer than 10 BIs found. Do not start "
        "the 10-prompt experiment yet."
    )

### Original cell 13
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
import hashlib

discovered = json.loads(
    (PHI_ROOT / "discovered_border_inputs.json").read_text()
)

assert discovered["model"] == PHI_MODEL
assert len(discovered["border_inputs"]) >= 10

PHI_TEST_DIR = PHI_ROOT / "smoke_test_v1"
PHI_TEST_DIR.mkdir(parents=True, exist_ok=True)

PHI_TEST_CONFIG = {
    "model": PHI_MODEL,
    "prompts": discovered["border_inputs"][:10],
    "n_samples": 100,
    "temperature": 0.0,
    "batch_size": 64,
    "input_fraction": 0.95,
    "seed": 0,
    "datasets_dir": str(OLD_DATA / "datasets"),
    "inference_settings": json.loads(
        (PHI_ROOT / "inference_settings.json").read_text()
    ),
}

PHI_TEST_JOBS = [
    {"name": "unchanged", "kind": "unchanged"},
    {
        "name": "noise_2pow_minus10",
        "kind": "random_noise",
        "scale": 2.0**-10,
    },
    {
        "name": "magnitude_2pow_minus10",
        "kind": "magnitude",
        "scale": 2.0**-10,
    },
    {
        "name": "random_pruning_2pow_minus10",
        "kind": "random",
        "scale": 2.0**-10,
    },
]

PHI_TEST_CONFIG_PATH = PHI_TEST_DIR / "config.json"

if PHI_TEST_CONFIG_PATH.exists():
    assert (
        json.loads(PHI_TEST_CONFIG_PATH.read_text())
        == PHI_TEST_CONFIG
    ), "Existing configuration differs."
else:
    PHI_TEST_CONFIG_PATH.write_text(
        json.dumps(PHI_TEST_CONFIG, indent=2)
    )

print("Selected prompts:")
for prompt in PHI_TEST_CONFIG["prompts"]:
    print(repr(prompt))

### Original cell 14
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
phi_worker_code = r'''
import argparse
import json
import random
from pathlib import Path

import numpy as np
import torch


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("action", choices=["edit", "sample"])
    parser.add_argument("--config", required=True)
    parser.add_argument("--job", required=True)
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    cfg = json.loads(Path(args.config).read_text())
    job = json.loads(Path(args.job).read_text())

    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])

    if args.action == "edit":
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from torch.nn.utils import prune

        model = AutoModelForCausalLM.from_pretrained(
            cfg["model"],
            torch_dtype=torch.float16,
        ).to("cuda:0")
        tokenizer = AutoTokenizer.from_pretrained(cfg["model"])

        affected_tensors = 0

        with torch.no_grad():
            if job["kind"] == "random_noise":
                for _, parameter in model.named_parameters():
                    if parameter.requires_grad:
                        parameter.add_(
                            torch.normal(
                                mean=0.0,
                                std=job["scale"],
                                size=parameter.shape,
                                device=parameter.device,
                                dtype=parameter.dtype,
                            )
                        )
                        affected_tensors += 1

            elif job["kind"] in ("magnitude", "random"):
                method = (
                    prune.l1_unstructured
                    if job["kind"] == "magnitude"
                    else prune.random_unstructured
                )

                for name, module in model.named_modules():
                    if "mlp" not in name:
                        continue

                    names = [
                        parameter_name
                        for parameter_name, _ in module.named_parameters(
                            recurse=False
                        )
                        if parameter_name == "weight"
                    ]

                    for parameter_name in names:
                        method(
                            module,
                            parameter_name,
                            amount=job["scale"],
                        )
                        prune.remove(module, parameter_name)
                        affected_tensors += 1

            else:
                raise ValueError("Unsupported editing action.")

        assert affected_tensors > 0, (
            "No parameter tensors matched the perturbation."
        )

        model.eval()
        model.save_pretrained(args.checkpoint)
        tokenizer.save_pretrained(args.checkpoint)

        print(
            "Checkpoint saved. Parameter tensors targeted:",
            affected_tensors,
            flush=True,
        )

    else:
        from vllm import LLM
        from track_llm_apis.util import load_lmsys_chat_1m
        from track_llm_apis.sampling.vllm_sampling import (
            vllm_inference_bi,
        )

        dataset = load_lmsys_chat_1m(
            use_cache=True,
            datasets_dir=Path(cfg["datasets_dir"]),
        )
        traffic = [
            row["conversation"][0]["content"]
            for row in dataset
        ]

        settings = cfg["inference_settings"]
        model_path = (
            cfg["model"]
            if job["kind"] == "unchanged"
            else args.checkpoint
        )

        llm = LLM(
            model=model_path,
            dtype=settings["dtype"],
            enforce_eager=settings["enforce_eager"],
            gpu_memory_utilization=settings["gpu_memory_utilization"],
            max_model_len=settings["max_model_len"],
            max_num_seqs=settings["max_num_seqs"],
            max_num_batched_tokens=settings["max_num_batched_tokens"],
            enable_chunked_prefill=True,
        )

        result = vllm_inference_bi(
            llm=llm,
            inputs=cfg["prompts"],
            traffic_prompts=traffic,
            n_samples=cfg["n_samples"],
            temperature=cfg["temperature"],
            input_fraction=cfg["input_fraction"],
            batch_size=cfg["batch_size"],
            max_tokens=1,
        )

        payload = {
            "config": cfg,
            "job": job,
            "counts": result.counts,
            "prompt_lengths": result.prompt_lengths,
            "output_tokens": result.output_tokens,
        }

        output = Path(args.output)
        temporary = output.with_suffix(".tmp")
        temporary.write_text(json.dumps(payload))
        temporary.replace(output)

        print("Collection saved:", output, flush=True)


if __name__ == "__main__":
    main()
'''

import ast

ast.parse(phi_worker_code)

PHI_WORKER = PHI_WORK / "perturbation_worker.py"
PHI_WORKER.write_text(phi_worker_code)

persistent_worker = PHI_TEST_DIR / "worker.py"
if persistent_worker.exists():
    assert persistent_worker.read_text() == phi_worker_code, (
        "Saved worker differs. Use a new experiment folder."
    )
else:
    persistent_worker.write_text(phi_worker_code)

(PHI_TEST_DIR / "worker_sha256.txt").write_text(
    hashlib.sha256(phi_worker_code.encode()).hexdigest()
)

print("Worker created; syntax checked.")

### Original cell 15
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import shutil
import time

def valid_phi_result(path, job):
    if not path.exists():
        return False

    try:
        saved = json.loads(path.read_text())

        if saved["config"] != PHI_TEST_CONFIG:
            return False
        if saved["job"] != job:
            return False

        return (
            set(saved["counts"]) == set(PHI_TEST_CONFIG["prompts"])
            and all(
                sum(saved["counts"][prompt].values())
                == PHI_TEST_CONFIG["n_samples"]
                for prompt in PHI_TEST_CONFIG["prompts"]
            )
        )
    except (KeyError, TypeError, ValueError):
        return False


def launch_phi_worker(action, job_path, checkpoint, output, log_name):
    arguments = [
        str(PHI_WORKER),
        action,
        "--config", str(PHI_TEST_CONFIG_PATH),
        "--job", str(job_path),
        "--checkpoint", str(checkpoint),
        "--output", str(output),
    ]

    launcher_code = (
        "import runpy\n"
        "import sys\n\n"
        'if __name__ == "__main__":\n'
        f"    sys.argv = {arguments!r}\n"
        f"    runpy.run_path({str(PHI_WORKER)!r}, "
        "run_name='__main__')\n"
    )

    launcher = PHI_WORK / "launch_worker.py"
    launcher.write_text(launcher_code)

    run_phi_script(launcher, log_name)


def collect_phi_job(job):
    output = PHI_TEST_DIR / f"{job['name']}.json"

    if valid_phi_result(output, job):
        print("Already complete; skipping:", job["name"])
        return

    job_path = PHI_TEST_DIR / f"{job['name']}_job.json"
    job_path.write_text(json.dumps(job, indent=2))

    checkpoint = PHI_WORK / "temporary_variant"
    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    started = time.monotonic()

    if job["kind"] != "unchanged":
        launch_phi_worker(
            "edit", job_path, checkpoint, output,
            f"{job['name']}_edit.log",
        )

    launch_phi_worker(
        "sample", job_path, checkpoint, output,
        f"{job['name']}_sample.log",
    )

    assert valid_phi_result(output, job)

    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    print(
        "Completed:", job["name"],
        "| minutes:",
        round((time.monotonic() - started) / 60, 1),
    )

### Original cell 16
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
collect_phi_job(PHI_TEST_JOBS[0])

phi_baseline = json.loads(
    (PHI_TEST_DIR / "unchanged.json").read_text()
)

variable_prompts = sum(
    len(counts) > 1
    for counts in phi_baseline["counts"].values()
)

print("Prompts with multiple observed outputs:", variable_prompts, "/ 10")

for prompt, counts in phi_baseline["counts"].items():
    print(repr(prompt), "->", counts)

### Original cell 17
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
for job in PHI_TEST_JOBS[1:]:
    print("\nVARIANT:", job["name"])
    collect_phi_job(job)

print("Three perturbation collections completed.")

### Original cell 18
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
for job in PHI_TEST_JOBS:
    path = PHI_TEST_DIR / f"{job['name']}.json"

    assert valid_phi_result(path, job), (
        f"Incomplete result: {job['name']}"
    )

    saved = json.loads(path.read_text())

    print(
        job["name"],
        "| prompts:", len(saved["counts"]),
        "| samples per prompt:",
        sorted({
            sum(counts.values())
            for counts in saved["counts"].values()
        }),
        "| variable-output prompts:",
        sum(
            len(counts) > 1
            for counts in saved["counts"].values()
        ),
    )

print("\nSaved:", PHI_TEST_DIR)

### Original cell 19
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

PHI_ANALYSIS_DIR = PHI_TEST_DIR / "analysis_independent_v1"
PHI_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

phi_collections = {}

for job in PHI_TEST_JOBS:
    path = PHI_TEST_DIR / f"{job['name']}.json"
    assert valid_phi_result(path, job)

    phi_collections[job["name"]] = json.loads(path.read_text())

PHI_ANALYSIS_SETTINGS = {
    "reference_samples": 50,
    "n_tests": 1000,
    "prompt_counts": [1, 3, 5, 10],
    "detection_counts": [1, 3, 10, 20, 50],
    "method": "independent empirical multinomial sampling",
}

manifest = {
    "settings": PHI_ANALYSIS_SETTINGS,
    "input_sha256": {
        job["name"]: hashlib.sha256(
            (PHI_TEST_DIR / f"{job['name']}.json").read_bytes()
        ).hexdigest()
        for job in PHI_TEST_JOBS
    },
}

manifest_path = PHI_ANALYSIS_DIR / "manifest.json"

if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))

print("Loaded four collections.")

### Original cell 20
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
def phi_auc(baseline, variant, n_prompts, n_detection, seed):
    rng = np.random.default_rng(seed)
    prompts = PHI_TEST_CONFIG["prompts"]
    n_reference = PHI_ANALYSIS_SETTINGS["reference_samples"]

    base_probabilities = {}
    variant_probabilities = {}

    for prompt in prompts:
        a_counts = baseline["counts"][prompt]
        b_counts = variant["counts"][prompt]
        tokens = sorted(set(a_counts) | set(b_counts))

        a = np.array(
            [a_counts.get(token, 0) for token in tokens],
            dtype=float,
        )
        b = np.array(
            [b_counts.get(token, 0) for token in tokens],
            dtype=float,
        )

        base_probabilities[prompt] = a / a.sum()
        variant_probabilities[prompt] = b / b.sum()

    def score(selected, left, right):
        distances = []

        for prompt in selected:
            reference = rng.multinomial(
                n_reference, left[prompt]
            ) / n_reference

            detection = rng.multinomial(
                n_detection, right[prompt]
            ) / n_detection

            distances.append(
                0.5 * np.abs(reference - detection).sum()
            )

        return float(np.mean(distances))

    h0, h1 = [], []

    for _ in range(PHI_ANALYSIS_SETTINGS["n_tests"]):
        selected = rng.choice(
            prompts, n_prompts, replace=False
        )

        h0.append(score(
            selected, base_probabilities, base_probabilities
        ))
        h0.append(score(
            selected, variant_probabilities, variant_probabilities
        ))
        h1.append(score(
            selected, base_probabilities, variant_probabilities
        ))

    return float(roc_auc_score(
        [0] * len(h0) + [1] * len(h1),
        h0 + h1,
    ))

print("Evaluation function ready.")

### Original cell 21
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
phi_baseline = phi_collections["unchanged"]

control_rows = []

for detection_samples in [3, 20, 50]:
    control_rows.append({
        "detection_samples_per_prompt": detection_samples,
        "unchanged_vs_unchanged_auc": phi_auc(
            phi_baseline,
            phi_baseline,
            n_prompts=10,
            n_detection=detection_samples,
            seed=12345 + detection_samples,
        ),
    })

phi_controls = pd.DataFrame(control_rows)
display(phi_controls)

phi_controls.to_csv(
    PHI_ANALYSIS_DIR / "unchanged_control.csv",
    index=False,
)

assert phi_controls[
    "unchanged_vs_unchanged_auc"
].between(0.45, 0.55).all(), (
    "Control is far from 0.5. Stop and share this output."
)

### Original cell 22
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
tasks = [
    (job, n_prompts, n_detection)
    for job in PHI_TEST_JOBS
    if job["kind"] != "unchanged"
    for n_prompts in PHI_ANALYSIS_SETTINGS["prompt_counts"]
    for n_detection in PHI_ANALYSIS_SETTINGS["detection_counts"]
]

rows = []

for job, n_prompts, n_detection in tqdm(
    tasks, desc="Phi B3IT evaluations"
):
    path = PHI_ANALYSIS_DIR / (
        f"{job['name']}_p{n_prompts}_d{n_detection}.json"
    )

    if path.exists():
        row = json.loads(path.read_text())
    else:
        key = f"phi-v1:{job['name']}:{n_prompts}:{n_detection}"
        seed = int(
            hashlib.sha256(key.encode()).hexdigest()[:8], 16
        )

        row = {
            "variant": job["name"],
            "kind": job["kind"],
            "scale": job["scale"],
            "n_prompts": n_prompts,
            "detection_samples_per_prompt": n_detection,
            "auc": phi_auc(
                phi_baseline,
                phi_collections[job["name"]],
                n_prompts,
                n_detection,
                seed,
            ),
        }

        temporary = path.with_suffix(".tmp")
        temporary.write_text(json.dumps(row, indent=2))
        temporary.replace(path)

    rows.append(row)

phi_sweep = pd.DataFrame(rows)
assert len(phi_sweep) == 60

phi_sweep.to_csv(
    PHI_ANALYSIS_DIR / "auc_per_variant.csv",
    index=False,
)

display(phi_sweep[
    (phi_sweep["n_prompts"] == 5)
    & (phi_sweep["detection_samples_per_prompt"] == 3)
][["variant", "auc"]])

### Original cell 23
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import matplotlib.pyplot as plt

phi_summary = (
    phi_sweep.groupby(
        ["n_prompts", "detection_samples_per_prompt"],
        as_index=False,
    )["auc"]
    .mean()
)

mean_input_tokens = np.mean([
    phi_baseline["prompt_lengths"][prompt]
    for prompt in PHI_TEST_CONFIG["prompts"]
])

# Pricing assumptions from the supplied repository.
phi_summary["yearly_cost"] = (
    8760
    * phi_summary["n_prompts"]
    * phi_summary["detection_samples_per_prompt"]
    * (3e-6 * mean_input_tokens + 12e-6)
)

colors = {
    1: "#E8C75A",
    3: "#FFB43B",
    5: "#F56600",
    10: "#963300",
}
sizes = dict(zip([1, 3, 10, 20, 50], [15, 25, 35, 48, 65]))

fig_phi, ax = plt.subplots(figsize=(9, 5.5))

for n_prompts, group in phi_summary.groupby("n_prompts"):
    group = group.sort_values("yearly_cost")

    ax.plot(
        group["yearly_cost"], group["auc"],
        color=colors[n_prompts],
        label=f"{n_prompts} prompts",
    )
    ax.scatter(
        group["yearly_cost"], group["auc"],
        color=colors[n_prompts],
        s=[
            sizes[n]
            for n in group["detection_samples_per_prompt"]
        ],
    )

ax.set_xscale("log")
ax.set_xlabel("Estimated annual monitoring cost ($)")
ax.set_ylabel("Mean ROC AUC over three variants")
ax.set_title(
    "Phi-4-mini — noise/pruning smoke test\n"
    "One variant per perturbation type at magnitude 2⁻¹⁰"
)
ax.grid(alpha=0.25)
ax.legend(title="Larger markers: more detection samples")

fig_phi.text(
    0.12, 0.015,
    "T4 / FP16; 10 Border Inputs; 100 recorded outputs per prompt;\n"
    "independent empirical bootstrap; 50 reference samples per test.",
    fontsize=8,
)
fig_phi.tight_layout(rect=[0, 0.07, 1, 1])

phi_summary.to_csv(
    PHI_ANALYSIS_DIR / "summary.csv", index=False
)
fig_phi.savefig(
    PHI_ANALYSIS_DIR / "phi_smoke_test.png",
    dpi=300,
    bbox_inches="tight",
)
fig_phi.savefig(
    PHI_ANALYSIS_DIR / "phi_smoke_test.pdf",
    bbox_inches="tight",
)

plt.show()
display(phi_summary)

### Original cell 24
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
display(phi_controls)

display(
    phi_sweep[
        (phi_sweep["n_prompts"] == 5)
        & (phi_sweep["detection_samples_per_prompt"] == 3)
    ][["variant", "auc"]]
)

### Original cell 25
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
import hashlib
import shutil
import time
import pandas as pd

PHI_EXPANDED_DIR = PHI_ROOT / "expanded_scales_100samples_v1"
PHI_EXPANDED_DIR.mkdir(parents=True, exist_ok=True)

PHI_EXPANDED_CONFIG = dict(PHI_TEST_CONFIG)
assert PHI_EXPANDED_CONFIG["n_samples"] == 100

PHI_EXPANDED_JOBS = [
    {"name": "unchanged", "kind": "unchanged"}
]

for exponent in [0, 5, 10, 15, 20, 25, 30]:
    PHI_EXPANDED_JOBS.append({
        "name": f"noise_2pow_minus{exponent}",
        "kind": "random_noise",
        "scale": 2.0 ** -exponent,
    })

for kind, prefix in [
    ("magnitude", "magnitude"),
    ("random", "random_pruning"),
]:
    for exponent in [0, 5, 10, 15, 20]:
        PHI_EXPANDED_JOBS.append({
            "name": f"{prefix}_2pow_minus{exponent}",
            "kind": kind,
            "scale": 2.0 ** -exponent,
        })

PHI_EXPANDED_CONFIG_PATH = PHI_EXPANDED_DIR / "config.json"

if PHI_EXPANDED_CONFIG_PATH.exists():
    assert (
        json.loads(PHI_EXPANDED_CONFIG_PATH.read_text())
        == PHI_EXPANDED_CONFIG
    )
else:
    PHI_EXPANDED_CONFIG_PATH.write_text(
        json.dumps(PHI_EXPANDED_CONFIG, indent=2)
    )

# Preserve the worker that successfully ran the smoke test.
source_worker = PHI_TEST_DIR / "worker.py"
assert source_worker.exists()

PHI_EXPANDED_WORKER = PHI_EXPANDED_DIR / "worker.py"

if PHI_EXPANDED_WORKER.exists():
    assert (
        PHI_EXPANDED_WORKER.read_bytes()
        == source_worker.read_bytes()
    )
else:
    shutil.copy2(source_worker, PHI_EXPANDED_WORKER)

expanded_manifest = {
    "config": PHI_EXPANDED_CONFIG,
    "jobs": PHI_EXPANDED_JOBS,
    "worker_sha256": hashlib.sha256(
        PHI_EXPANDED_WORKER.read_bytes()
    ).hexdigest(),
}

manifest_path = PHI_EXPANDED_DIR / "manifest.json"

if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == expanded_manifest
else:
    manifest_path.write_text(
        json.dumps(expanded_manifest, indent=2)
    )

print("Perturbation variants:", len(PHI_EXPANDED_JOBS) - 1)
print("Total versions:", len(PHI_EXPANDED_JOBS))
print("Results:", PHI_EXPANDED_DIR)

### Original cell 26
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
def valid_phi_expanded(path, job):
    if not path.exists():
        return False

    try:
        saved = json.loads(path.read_text())

        return (
            saved["config"] == PHI_EXPANDED_CONFIG
            and saved["job"] == job
            and set(saved["counts"])
                == set(PHI_EXPANDED_CONFIG["prompts"])
            and all(
                sum(saved["counts"][prompt].values())
                == PHI_EXPANDED_CONFIG["n_samples"]
                for prompt in PHI_EXPANDED_CONFIG["prompts"]
            )
        )
    except (KeyError, TypeError, ValueError):
        return False


for job in PHI_EXPANDED_JOBS:
    source = PHI_TEST_DIR / f"{job['name']}.json"
    destination = PHI_EXPANDED_DIR / f"{job['name']}.json"

    if valid_phi_expanded(destination, job):
        print("Already available:", job["name"])
    elif valid_phi_expanded(source, job):
        shutil.copy2(source, destination)
        print("Reused:", job["name"])

remaining = [
    job for job in PHI_EXPANDED_JOBS
    if not valid_phi_expanded(
        PHI_EXPANDED_DIR / f"{job['name']}.json", job
    )
]

print("\nRemaining versions to collect:", len(remaining))

### Original cell 27
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
def launch_phi_expanded(
    action, job_path, checkpoint, output, log_name
):
    arguments = [
        str(PHI_EXPANDED_WORKER),
        action,
        "--config", str(PHI_EXPANDED_CONFIG_PATH),
        "--job", str(job_path),
        "--checkpoint", str(checkpoint),
        "--output", str(output),
    ]

    launcher_code = (
        "import runpy\n"
        "import sys\n\n"
        'if __name__ == "__main__":\n'
        f"    sys.argv = {arguments!r}\n"
        f"    runpy.run_path({str(PHI_EXPANDED_WORKER)!r}, "
        "run_name='__main__')\n"
    )

    launcher = PHI_WORK / "launch_expanded.py"
    launcher.write_text(launcher_code)

    run_phi_script(launcher, log_name)


def collect_phi_expanded(job):
    output = PHI_EXPANDED_DIR / f"{job['name']}.json"

    if valid_phi_expanded(output, job):
        print("Already complete; skipping:", job["name"])
        return

    job_path = PHI_EXPANDED_DIR / f"{job['name']}_job.json"
    job_path.write_text(json.dumps(job, indent=2))

    checkpoint = PHI_WORK / "temporary_expanded_variant"
    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    started = time.monotonic()

    if job["kind"] != "unchanged":
        launch_phi_expanded(
            "edit", job_path, checkpoint, output,
            f"expanded_{job['name']}_edit.log",
        )

    launch_phi_expanded(
        "sample", job_path, checkpoint, output,
        f"expanded_{job['name']}_sample.log",
    )

    assert valid_phi_expanded(output, job)

    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    elapsed = time.monotonic() - started

    (PHI_EXPANDED_DIR / f"{job['name']}_timing.json").write_text(
        json.dumps({"elapsed_seconds": elapsed}, indent=2)
    )

    print(
        "Saved and verified:", job["name"],
        "| minutes:", round(elapsed / 60, 1),
    )

### Original cell 28
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
for index, job in enumerate(PHI_EXPANDED_JOBS, start=1):
    print(
        f"\n[{index}/{len(PHI_EXPANDED_JOBS)}]",
        job["name"],
    )
    collect_phi_expanded(job)

print("\nExpanded collection completed.")

### Original cell 29
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
verification_rows = []

for job in PHI_EXPANDED_JOBS:
    path = PHI_EXPANDED_DIR / f"{job['name']}.json"
    complete = valid_phi_expanded(path, job)

    row = {
        "variant": job["name"],
        "complete": complete,
    }

    if complete:
        saved = json.loads(path.read_text())
        row["variable_output_prompts"] = sum(
            len(counts) > 1
            for counts in saved["counts"].values()
        )

    verification_rows.append(row)

phi_expanded_verification = pd.DataFrame(verification_rows)
display(phi_expanded_verification)

assert phi_expanded_verification["complete"].all(), (
    "Some versions are incomplete. Rerun Cell 28."
)

phi_expanded_verification.to_csv(
    PHI_EXPANDED_DIR / "verification.csv",
    index=False,
)

print("Verified: 18 versions × 10 prompts × 100 outputs.")

### Original cell 30
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from tqdm.auto import tqdm

assert PHI_EXPANDED_CONFIG == PHI_TEST_CONFIG

PHI_EXPANDED_ANALYSIS = (
    PHI_EXPANDED_DIR / "analysis_independent_v1"
)
PHI_EXPANDED_ANALYSIS.mkdir(parents=True, exist_ok=True)

expanded_collections = {
    job["name"]: json.loads(
        (PHI_EXPANDED_DIR / f"{job['name']}.json").read_text()
    )
    for job in PHI_EXPANDED_JOBS
}

expanded_baseline = expanded_collections["unchanged"]

analysis_manifest = {
    "settings": PHI_ANALYSIS_SETTINGS,
    "inputs": {
        job["name"]: hashlib.sha256(
            (PHI_EXPANDED_DIR / f"{job['name']}.json").read_bytes()
        ).hexdigest()
        for job in PHI_EXPANDED_JOBS
    },
}

path = PHI_EXPANDED_ANALYSIS / "manifest.json"
if path.exists():
    assert json.loads(path.read_text()) == analysis_manifest
else:
    path.write_text(json.dumps(analysis_manifest, indent=2))

tasks = [
    (job, prompts, samples)
    for job in PHI_EXPANDED_JOBS
    if job["kind"] != "unchanged"
    for prompts in PHI_ANALYSIS_SETTINGS["prompt_counts"]
    for samples in PHI_ANALYSIS_SETTINGS["detection_counts"]
]

rows = []

for job, prompts, samples in tqdm(
    tasks, desc="Expanded Phi analysis"
):
    path = PHI_EXPANDED_ANALYSIS / (
        f"{job['name']}_p{prompts}_d{samples}.json"
    )

    if path.exists():
        row = json.loads(path.read_text())
    else:
        key = f"phi-expanded-v1:{job['name']}:{prompts}:{samples}"
        seed = int(
            hashlib.sha256(key.encode()).hexdigest()[:8], 16
        )

        row = {
            "variant": job["name"],
            "kind": job["kind"],
            "scale": job["scale"],
            "n_prompts": prompts,
            "detection_samples_per_prompt": samples,
            "auc": phi_auc(
                expanded_baseline,
                expanded_collections[job["name"]],
                prompts,
                samples,
                seed,
            ),
        }

        temporary = path.with_suffix(".tmp")
        temporary.write_text(json.dumps(row, indent=2))
        temporary.replace(path)

    rows.append(row)

phi_expanded_sweep = pd.DataFrame(rows)
assert len(phi_expanded_sweep) == 340

phi_expanded_sweep.to_csv(
    PHI_EXPANDED_ANALYSIS / "auc_per_variant.csv",
    index=False,
)

display(
    phi_expanded_sweep[
        (phi_expanded_sweep["n_prompts"] == 5)
        & (
            phi_expanded_sweep["detection_samples_per_prompt"]
            == 3
        )
    ][["kind", "scale", "auc"]]
)

### Original cell 31
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

phi_expanded_summary = (
    phi_expanded_sweep.groupby(
        ["n_prompts", "detection_samples_per_prompt"],
        as_index=False,
    )
    .agg(
        auc=("auc", "mean"),
        n_variants=("variant", "nunique"),
    )
)

assert len(phi_expanded_summary) == 20
assert (phi_expanded_summary["n_variants"] == 17).all()

mean_input_tokens = np.mean([
    expanded_baseline["prompt_lengths"][prompt]
    for prompt in PHI_EXPANDED_CONFIG["prompts"]
])

phi_expanded_summary["yearly_cost"] = (
    8760
    * phi_expanded_summary["n_prompts"]
    * phi_expanded_summary["detection_samples_per_prompt"]
    * (3e-6 * mean_input_tokens + 12e-6)
)

colors = {
    1: "#E8C75A",
    3: "#FFB43B",
    5: "#F56600",
    10: "#963300",
}
sizes = dict(zip([1, 3, 10, 20, 50], [15, 25, 35, 48, 65]))

fig_phi_expanded, ax = plt.subplots(figsize=(9, 5.5))

for prompts, group in phi_expanded_summary.groupby("n_prompts"):
    group = group.sort_values("yearly_cost")

    ax.plot(
        group["yearly_cost"], group["auc"],
        color=colors[prompts],
        label=f"{prompts} prompts",
    )
    ax.scatter(
        group["yearly_cost"], group["auc"],
        color=colors[prompts],
        s=[
            sizes[n]
            for n in group["detection_samples_per_prompt"]
        ],
    )

ax.set_xscale("log")
ax.set_xlabel("Estimated annual monitoring cost ($)")
ax.set_ylabel("Mean ROC AUC over 17 variants")
ax.set_title(
    "Partial Figure 1 — Phi-4-mini\n"
    "Gaussian noise, magnitude pruning and random pruning"
)
ax.grid(alpha=0.25)
ax.legend(title="Larger markers: more detection samples")

fig_phi_expanded.text(
    0.12, 0.015,
    "T4 / FP16; 10 Border Inputs; 100 recorded outputs per prompt;\n"
    "independent empirical bootstrap; 50 reference samples per test.\n"
    "Costs use the supplied repository's pricing assumptions.",
    fontsize=8,
)
fig_phi_expanded.tight_layout(rect=[0, 0.09, 1, 1])

phi_expanded_summary.to_csv(
    PHI_EXPANDED_ANALYSIS / "summary.csv",
    index=False,
)
fig_phi_expanded.savefig(
    PHI_EXPANDED_ANALYSIS / "phi_expanded_figure1.png",
    dpi=300,
    bbox_inches="tight",
)
fig_phi_expanded.savefig(
    PHI_EXPANDED_ANALYSIS / "phi_expanded_figure1.pdf",
    bbox_inches="tight",
)

plt.show()
display(phi_expanded_summary)

### Original cell 32
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd

PHI_BASELINE_CHECK_DIR = PHI_ROOT / "independent_baseline_check_v1"
PHI_BASELINE_CHECK_DIR.mkdir(parents=True, exist_ok=True)

phi_first_baseline = json.loads(
    (PHI_EXPANDED_DIR / "unchanged.json").read_text()
)

PHI_SECOND_CONFIG = dict(phi_first_baseline["config"])
PHI_SECOND_CONFIG["seed"] = (
    phi_first_baseline["config"]["seed"] + 1
)

PHI_SECOND_JOB = {
    "name": "unchanged_independent",
    "kind": "unchanged",
}

PHI_SECOND_CONFIG_PATH = PHI_BASELINE_CHECK_DIR / "config.json"
PHI_SECOND_JOB_PATH = PHI_BASELINE_CHECK_DIR / "job.json"
PHI_SECOND_OUTPUT = (
    PHI_BASELINE_CHECK_DIR / "unchanged_independent.json"
)

for path, payload in [
    (PHI_SECOND_CONFIG_PATH, PHI_SECOND_CONFIG),
    (PHI_SECOND_JOB_PATH, PHI_SECOND_JOB),
]:
    if path.exists():
        assert json.loads(path.read_text()) == payload, (
            f"Existing settings differ: {path}"
        )
    else:
        path.write_text(json.dumps(payload, indent=2))

print("First seed:", phi_first_baseline["config"]["seed"])
print("Second seed:", PHI_SECOND_CONFIG["seed"])
print("Prompts:", len(PHI_SECOND_CONFIG["prompts"]))
print("Samples per prompt:", PHI_SECOND_CONFIG["n_samples"])

### Original cell 33
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
def valid_phi_second_baseline():
    if not PHI_SECOND_OUTPUT.exists():
        return False

    try:
        saved = json.loads(PHI_SECOND_OUTPUT.read_text())

        return (
            saved["config"] == PHI_SECOND_CONFIG
            and saved["job"] == PHI_SECOND_JOB
            and set(saved["counts"])
                == set(PHI_SECOND_CONFIG["prompts"])
            and all(
                sum(saved["counts"][prompt].values())
                == PHI_SECOND_CONFIG["n_samples"]
                for prompt in PHI_SECOND_CONFIG["prompts"]
            )
        )
    except (KeyError, ValueError, TypeError):
        return False


if valid_phi_second_baseline():
    print("Second baseline already complete; skipping.")
else:
    worker = PHI_EXPANDED_DIR / "worker.py"

    manifest = json.loads(
        (PHI_EXPANDED_DIR / "manifest.json").read_text()
    )
    assert hashlib.sha256(
        worker.read_bytes()
    ).hexdigest() == manifest["worker_sha256"]

    arguments = [
        str(worker),
        "sample",
        "--config", str(PHI_SECOND_CONFIG_PATH),
        "--job", str(PHI_SECOND_JOB_PATH),
        "--checkpoint", str(PHI_WORK / "unused_checkpoint"),
        "--output", str(PHI_SECOND_OUTPUT),
    ]

    launcher = PHI_WORK / "launch_second_baseline.py"
    launcher.write_text(
        "import runpy\n"
        "import sys\n\n"
        'if __name__ == "__main__":\n'
        f"    sys.argv = {arguments!r}\n"
        f"    runpy.run_path({str(worker)!r}, "
        "run_name='__main__')\n"
    )

    run_phi_script(
        launcher,
        "independent_baseline_collection.log",
    )

assert valid_phi_second_baseline()
print("Second Phi baseline saved and verified.")

### Original cell 34
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
phi_second_baseline = json.loads(
    PHI_SECOND_OUTPUT.read_text()
)

phi_baseline_pairs = []
tv_rows = []

for index, prompt in enumerate(PHI_SECOND_CONFIG["prompts"]):
    first_counts = phi_first_baseline["counts"][prompt]
    second_counts = phi_second_baseline["counts"][prompt]

    tokens = sorted(set(first_counts) | set(second_counts))

    first = np.array(
        [first_counts.get(token, 0) for token in tokens],
        dtype=np.int64,
    )
    second = np.array(
        [second_counts.get(token, 0) for token in tokens],
        dtype=np.int64,
    )

    tv = 0.5 * np.abs(
        first / first.sum() - second / second.sum()
    ).sum()

    phi_baseline_pairs.append((first, second))

    tv_rows.append({
        "prompt_index": index,
        "prompt": repr(prompt),
        "first_samples": int(first.sum()),
        "second_samples": int(second.sum()),
        "tv_distance": float(tv),
    })

phi_baseline_tv = pd.DataFrame(tv_rows)
phi_observed_mean_tv = float(
    phi_baseline_tv["tv_distance"].mean()
)

display(phi_baseline_tv)
print("Observed mean TV:", phi_observed_mean_tv)

phi_baseline_tv.to_csv(
    PHI_BASELINE_CHECK_DIR / "tv_between_runs.csv",
    index=False,
)

### Original cell 35
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from tqdm.auto import tqdm

rng = np.random.default_rng(2026)
N_PERMUTATIONS = 5000

pooled_data = []

for first, second in phi_baseline_pairs:
    combined = first + second

    pooled_data.append({
        "pool": np.repeat(np.arange(len(combined)), combined),
        "counts": combined,
        "n_first": int(first.sum()),
        "n_second": int(second.sum()),
    })

phi_null_mean_tv = np.empty(N_PERMUTATIONS)

for iteration in tqdm(
    range(N_PERMUTATIONS),
    desc="Phi baseline permutation check",
):
    distances = []

    for item in pooled_data:
        indices = rng.choice(
            len(item["pool"]),
            item["n_first"],
            replace=False,
        )

        first_permuted = np.bincount(
            item["pool"][indices],
            minlength=len(item["counts"]),
        )
        second_permuted = (
            item["counts"] - first_permuted
        )

        distances.append(
            0.5 * np.abs(
                first_permuted / item["n_first"]
                - second_permuted / item["n_second"]
            ).sum()
        )

    phi_null_mean_tv[iteration] = np.mean(distances)

phi_p_value = (
    1 + np.count_nonzero(
        phi_null_mean_tv >= phi_observed_mean_tv
    )
) / (N_PERMUTATIONS + 1)

phi_check_report = {
    "observed_mean_tv": phi_observed_mean_tv,
    "null_mean_tv": float(phi_null_mean_tv.mean()),
    "null_95th_percentile": float(
        np.quantile(phi_null_mean_tv, 0.95)
    ),
    "permutation_p_value": float(phi_p_value),
    "n_permutations": N_PERMUTATIONS,
}

(PHI_BASELINE_CHECK_DIR / "permutation_report.json").write_text(
    json.dumps(phi_check_report, indent=2)
)
np.save(
    PHI_BASELINE_CHECK_DIR / "permutation_null.npy",
    phi_null_mean_tv,
)

display(pd.DataFrame([phi_check_report]))

### Original cell 36
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import matplotlib.pyplot as plt

fig_phi_check, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(
    phi_null_mean_tv,
    bins=40,
    color="#8DB9DE",
    edgecolor="white",
    label="Shared-distribution permutation results",
)

ax.axvline(
    phi_observed_mean_tv,
    color="#B53324",
    linewidth=2,
    label=f"Observed mean TV = {phi_observed_mean_tv:.4f}",
)

ax.set_xlabel("Mean TV across 10 prompts")
ax.set_ylabel("Permutation count")
ax.set_title("Unchanged Phi-4-mini — two separate collections")
ax.legend()
fig_phi_check.tight_layout()

fig_phi_check.savefig(
    PHI_BASELINE_CHECK_DIR / "baseline_check.png",
    dpi=300,
)
fig_phi_check.savefig(
    PHI_BASELINE_CHECK_DIR / "baseline_check.pdf",
)

plt.show()
print("Permutation p-value:", phi_p_value)
print("Saved:", PHI_BASELINE_CHECK_DIR)

### Original cell 37
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
import hashlib
import shutil
import time
import pandas as pd

PHI_LARGE_DIR = PHI_ROOT / "expanded_scales_1000samples_v1"
PHI_LARGE_DIR.mkdir(parents=True, exist_ok=True)

PHI_LARGE_CONFIG = json.loads(json.dumps(PHI_EXPANDED_CONFIG))
PHI_LARGE_CONFIG["n_samples"] = 1000

PHI_LARGE_JOBS = [
    dict(job) for job in PHI_EXPANDED_JOBS
]

PHI_LARGE_CONFIG_PATH = PHI_LARGE_DIR / "config.json"

if PHI_LARGE_CONFIG_PATH.exists():
    assert (
        json.loads(PHI_LARGE_CONFIG_PATH.read_text())
        == PHI_LARGE_CONFIG
    ), "Existing collection settings differ."
else:
    PHI_LARGE_CONFIG_PATH.write_text(
        json.dumps(PHI_LARGE_CONFIG, indent=2)
    )

source_worker = PHI_EXPANDED_DIR / "worker.py"
PHI_LARGE_WORKER = PHI_LARGE_DIR / "worker.py"

if PHI_LARGE_WORKER.exists():
    assert PHI_LARGE_WORKER.read_bytes() == source_worker.read_bytes()
else:
    shutil.copy2(source_worker, PHI_LARGE_WORKER)

large_manifest = {
    "config": PHI_LARGE_CONFIG,
    "jobs": PHI_LARGE_JOBS,
    "worker_sha256": hashlib.sha256(
        PHI_LARGE_WORKER.read_bytes()
    ).hexdigest(),
}

manifest_path = PHI_LARGE_DIR / "manifest.json"

if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == large_manifest
else:
    manifest_path.write_text(json.dumps(large_manifest, indent=2))

print("Model:", PHI_LARGE_CONFIG["model"])
print("Prompts:", len(PHI_LARGE_CONFIG["prompts"]))
print("Samples per prompt:", PHI_LARGE_CONFIG["n_samples"])
print("Model versions:", len(PHI_LARGE_JOBS))
print("Results:", PHI_LARGE_DIR)

### Original cell 38
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
def valid_phi_large(path, job):
    if not path.exists():
        return False

    try:
        saved = json.loads(path.read_text())

        return (
            saved["config"] == PHI_LARGE_CONFIG
            and saved["job"] == job
            and set(saved["counts"])
                == set(PHI_LARGE_CONFIG["prompts"])
            and all(
                sum(saved["counts"][prompt].values())
                == PHI_LARGE_CONFIG["n_samples"]
                for prompt in PHI_LARGE_CONFIG["prompts"]
            )
        )
    except (KeyError, ValueError, TypeError):
        return False

print("Verification function ready.")

### Original cell 39
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
def launch_phi_large(
    action, job_path, checkpoint, output, log_name
):
    arguments = [
        str(PHI_LARGE_WORKER),
        action,
        "--config", str(PHI_LARGE_CONFIG_PATH),
        "--job", str(job_path),
        "--checkpoint", str(checkpoint),
        "--output", str(output),
    ]

    launcher = PHI_WORK / "launch_large_collection.py"
    launcher.write_text(
        "import runpy\n"
        "import sys\n\n"
        'if __name__ == "__main__":\n'
        f"    sys.argv = {arguments!r}\n"
        f"    runpy.run_path({str(PHI_LARGE_WORKER)!r}, "
        "run_name='__main__')\n"
    )

    run_phi_script(launcher, log_name)


def collect_phi_large(job):
    output = PHI_LARGE_DIR / f"{job['name']}.json"

    if valid_phi_large(output, job):
        print("Already complete; skipping:", job["name"])
        return

    job_path = PHI_LARGE_DIR / f"{job['name']}_job.json"
    job_path.write_text(json.dumps(job, indent=2))

    checkpoint = PHI_WORK / "temporary_large_variant"
    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    started = time.monotonic()

    if job["kind"] != "unchanged":
        launch_phi_large(
            "edit", job_path, checkpoint, output,
            f"large_{job['name']}_edit.log",
        )

    launch_phi_large(
        "sample", job_path, checkpoint, output,
        f"large_{job['name']}_sample.log",
    )

    assert valid_phi_large(output, job)

    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    elapsed = time.monotonic() - started
    (PHI_LARGE_DIR / f"{job['name']}_timing.json").write_text(
        json.dumps({"elapsed_seconds": elapsed}, indent=2)
    )

    print(
        "Saved:", job["name"],
        "| minutes:", round(elapsed / 60, 1),
    )

### Original cell 40
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
baseline_job = next(
    job for job in PHI_LARGE_JOBS
    if job["kind"] == "unchanged"
)

collect_phi_large(baseline_job)

timing_path = PHI_LARGE_DIR / "unchanged_timing.json"

if timing_path.exists():
    seconds = json.loads(timing_path.read_text())["elapsed_seconds"]

    print("\nBaseline minutes:", round(seconds / 60, 1))
    print(
        "Rough time for 18 similar collections:",
        round(18 * seconds / 3600, 2),
        "hours.",
    )
    print(
        "Variant editing and different inference speeds "
        "can increase or change this estimate."
    )

### Original cell 41
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
for index, job in enumerate(PHI_LARGE_JOBS, start=1):
    print(
        f"\n[{index}/{len(PHI_LARGE_JOBS)}]",
        job["name"],
    )
    collect_phi_large(job)

print("\nAll collections completed.")

### Original cell 42
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### Original cell 43
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import json
import subprocess

root = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
)
folder = root / "expanded_scales_1000samples_v1"

assert folder.exists(), (
    "Mount Google Drive first, then rerun this cell."
)

manifest = json.loads((folder / "manifest.json").read_text())
config = manifest["config"]
jobs = manifest["jobs"]

completed = []
pending = []

for job in jobs:
    path = folder / f"{job['name']}.json"
    valid = False

    if path.exists():
        try:
            saved = json.loads(path.read_text())
            valid = (
                saved["config"] == config
                and saved["job"] == job
                and set(saved["counts"]) == set(config["prompts"])
                and all(
                    sum(saved["counts"][p].values())
                    == config["n_samples"]
                    for p in config["prompts"]
                )
            )
        except (KeyError, ValueError, TypeError):
            pass

    (completed if valid else pending).append(job["name"])

print("COMPLETED:", len(completed), "/", len(jobs))
for name in completed:
    print("  ✓", name)

print("\nPENDING:", len(pending))
for name in pending:
    print("  -", name)

print("\nGPU STATUS:")
subprocess.run(["nvidia-smi"], check=False)

print(
    "\nLocal environment exists:",
    Path(
        "/content/b3it_phi4mini_repo/"
        "track-llm-apis/.venv/bin/python"
    ).exists(),
)

### Original cell 44
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Original cell 45
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
from pathlib import Path
import pandas as pd

PHI_ROOT = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
)

if not PHI_ROOT.exists():
    raise FileNotFoundError(
        f"Folder not found: {PHI_ROOT}\n"
        "Check that you mounted the correct Drive account."
    )

rows = []

for folder_name in [
    "expanded_scales_v1",
    "expanded_scales_1000samples_v1",
]:
    folder = PHI_ROOT / folder_name

    if not folder.exists():
        print("Folder not found:", folder_name)
        continue

    for path in sorted(folder.glob("*.json")):
        try:
            saved = json.loads(path.read_text())
        except (OSError, ValueError) as error:
            print("Could not read:", path.name, str(error))
            continue

        if not isinstance(saved, dict):
            continue

        counts = saved.get("counts")
        if not isinstance(counts, dict) or not counts:
            continue

        valid_counts = all(
            isinstance(outputs, dict)
            and bool(outputs)
            and all(
                type(number) is int and number >= 0
                for number in outputs.values()
            )
            for outputs in counts.values()
        )

        rows.append({
            "collection": folder_name,
            "file": path.name,
            "model": saved.get("config", {}).get("model"),
            "prompts": len(counts),
            "outputs_per_prompt": (
                sorted({
                    sum(outputs.values())
                    for outputs in counts.values()
                })
                if valid_counts else "Invalid counts"
            ),
        })

phi_inventory = pd.DataFrame(rows)

if phi_inventory.empty:
    print("No output-count files found in these folders.")
    print("\nAvailable folders:")
    for folder in sorted(PHI_ROOT.iterdir()):
        if folder.is_dir():
            print(folder.name)
else:
    display(phi_inventory)

### Original cell 46
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []

# Search only within your Phi project folder.
for path in sorted(PHI_ROOT.rglob("*.json")):
    try:
        saved = json.loads(path.read_text())
    except (OSError, ValueError):
        continue

    if not isinstance(saved, dict):
        continue

    counts = saved.get("counts")
    if not isinstance(counts, dict) or not counts:
        continue

    if not all(
        isinstance(outputs, dict)
        and bool(outputs)
        and all(
            type(value) is int and value >= 0
            for value in outputs.values()
        )
        for outputs in counts.values()
    ):
        continue

    rows.append({
        "folder": str(path.parent.relative_to(PHI_ROOT)),
        "file": path.name,
        "model": saved.get("config", {}).get("model"),
        "prompts": len(counts),
        "outputs_per_prompt": sorted({
            sum(outputs.values())
            for outputs in counts.values()
        }),
    })

phi_files = pd.DataFrame(rows)

if phi_files.empty:
    print("No output-count files found.")
else:
    with pd.option_context(
        "display.max_rows", None,
        "display.max_colwidth", 100,
    ):
        display(phi_files)

    print("\nOUTPUT FILES PER FOLDER")
    display(
        phi_files.groupby("folder")
        .size()
        .rename("number_of_files")
        .reset_index()
    )

### Original cell 47
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
from pathlib import Path

PHI_COLLECTION = PHI_ROOT / "expanded_scales_100samples_v1"
PHI_MODEL = "microsoft/Phi-4-mini-instruct"

phi_collections = {}

for path in sorted(PHI_COLLECTION.glob("*.json")):
    saved = json.loads(path.read_text())

    if not isinstance(saved, dict):
        continue
    if not isinstance(saved.get("counts"), dict):
        continue

    assert saved["config"]["model"] == PHI_MODEL, path.name
    phi_collections[path.stem] = saved

assert len(phi_collections) == 18
assert "unchanged" in phi_collections

phi_baseline = phi_collections["unchanged"]
PHI_PROMPTS = phi_baseline["config"]["prompts"]

assert len(PHI_PROMPTS) == len(set(PHI_PROMPTS)) == 10

phi_families = {}

for version, saved in phi_collections.items():
    assert set(saved["counts"]) == set(PHI_PROMPTS), version

    for counts in saved["counts"].values():
        assert all(
            type(count) is int and count >= 0
            for count in counts.values()
        ), version
        assert sum(counts.values()) == 100, version

    if version != "unchanged":
        kind = saved["job"]["kind"]
        phi_families.setdefault(kind, []).append(version)

assert {
    kind: len(versions)
    for kind, versions in phi_families.items()
} == {
    "magnitude": 5,
    "random": 5,
    "random_noise": 7,
}

PHI_SPLITS = {}

for held_out in sorted(phi_families):
    PHI_SPLITS[held_out] = {
        "selection": sorted(
            version
            for kind, versions in phi_families.items()
            if kind != held_out
            for version in versions
        ),
        "evaluation": sorted(phi_families[held_out]),
    }

PHI_PROTOCOL = {
    "model": PHI_MODEL,
    "seeds": [101, 202, 303, 404, 505],
    "candidate_prompts": PHI_PROMPTS,
    "selection_pool_size": 50,
    "evaluation_pool_size": 50,
    "selected_prompts": 3,
    "detection_samples_per_prompt": 10,
    "reference_samples_per_prompt": 50,
    "simulation_tests": 1000,
    "detection_requests_per_test": 30,
    "strategies": [
        "performance_based",
        "entropy_based",
        "uniform_random_subset",
    ],
    "selection_aggregation": (
        "Mean prompt AUC within each selection perturbation type, "
        "then equal average across selection types"
    ),
    "evaluation_aggregation": (
        "Mean group AUC across variants within each held-out type"
    ),
    "held_out_splits": PHI_SPLITS,
    "primary_metric": "Held-out ROC AUC per perturbation type",
    "resampling": (
        "Split recorded outputs without replacement. "
        "Within each pool, independently bootstrap reference "
        "and detection samples with replacement."
    ),
    "cost_scope": (
        "Detection requests only; discovery, selection and "
        "reference collection costs excluded."
    ),
    "limitations": [
        "Ten previously collected candidate Border Inputs",
        "Only 50 recorded outputs per selection/evaluation pool",
        "Repeated splits reuse the same original collection",
        "Entropy baseline is not a verified original-repository rule",
        "Different recorded-pool size from the Qwen experiment",
    ],
}

PHI_REPEAT_DIR = PHI_COLLECTION / "extension_selection_repeated_v1"
PHI_REPEAT_DIR.mkdir(parents=True, exist_ok=True)

phi_protocol_path = PHI_REPEAT_DIR / "protocol.json"

if phi_protocol_path.exists():
    assert json.loads(phi_protocol_path.read_text()) == PHI_PROTOCOL, (
        "A different protocol exists. Use a new experiment folder."
    )
    print("Matching Phi protocol already saved.")
else:
    phi_protocol_path.write_text(
        json.dumps(PHI_PROTOCOL, indent=2, ensure_ascii=False)
    )
    print("Phi protocol saved.")

print("Verified: 18 versions × 10 prompts × 100 outputs.")
print("Pools: 50 selection + 50 evaluation outputs.")
print("Detection budget: 3 × 10 = 30 requests per test.")
print("Seeds:", PHI_PROTOCOL["seeds"])
print("Saved:", phi_protocol_path)

### Original cell 48
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def run_phi_extension():
    protocol = json.loads(phi_protocol_path.read_text())

    assert protocol == PHI_PROTOCOL

    prompts = protocol["candidate_prompts"]
    seeds = protocol["seeds"]
    splits = protocol["held_out_splits"]

    n_selected = protocol["selected_prompts"]
    n_detection = protocol["detection_samples_per_prompt"]
    n_reference = protocol["reference_samples_per_prompt"]
    n_tests = protocol["simulation_tests"]
    pool_size = protocol["selection_pool_size"]

    groups = list(combinations(range(len(prompts)), n_selected))
    group_indices = np.array(groups)

    assert len(groups) == 120

    labels = np.concatenate([
        np.zeros(n_tests),
        np.ones(n_tests),
    ])

    def split_outputs(seed):
        rng = np.random.default_rng(seed)
        selection, evaluation = {}, {}

        for version in sorted(phi_collections):
            selection[version] = {}
            evaluation[version] = {}

            for prompt in prompts:
                counts = phi_collections[version]["counts"][prompt]
                tokens = sorted(counts)

                frequencies = np.array(
                    [counts[token] for token in tokens],
                    dtype=np.int64,
                )
                assert frequencies.sum() == 2 * pool_size

                selected = rng.multivariate_hypergeometric(
                    frequencies, pool_size
                )
                remaining = frequencies - selected

                selection[version][prompt] = {
                    token: int(count)
                    for token, count in zip(tokens, selected)
                    if count > 0
                }
                evaluation[version][prompt] = {
                    token: int(count)
                    for token, count in zip(tokens, remaining)
                    if count > 0
                }

        return selection, evaluation

    def simulate_scores(pool, variant, rng):
        null_scores, changed_scores = [], []

        for prompt in prompts:
            original = pool["unchanged"][prompt]
            changed = pool[variant][prompt]

            tokens = sorted(set(original) | set(changed))

            p0 = np.array(
                [original.get(token, 0) for token in tokens],
                dtype=float,
            )
            p1 = np.array(
                [changed.get(token, 0) for token in tokens],
                dtype=float,
            )
            p0 /= p0.sum()
            p1 /= p1.sum()

            # Independent reference and detection bootstrap draws.
            ref0 = rng.multinomial(
                n_reference, p0, size=n_tests
            ) / n_reference
            det0 = rng.multinomial(
                n_detection, p0, size=n_tests
            ) / n_detection

            ref1 = rng.multinomial(
                n_reference, p0, size=n_tests
            ) / n_reference
            det1 = rng.multinomial(
                n_detection, p1, size=n_tests
            ) / n_detection

            null_scores.append(
                0.5 * np.abs(ref0 - det0).sum(axis=1)
            )
            changed_scores.append(
                0.5 * np.abs(ref1 - det1).sum(axis=1)
            )

        return np.array(null_scores), np.array(changed_scores)

    def calculate_auc(null, changed):
        return float(roc_auc_score(
            labels,
            np.concatenate([null, changed]),
        ))

    for seed in seeds:
        checkpoint = PHI_REPEAT_DIR / f"seed_{seed}.json"

        if checkpoint.exists():
            saved = json.loads(checkpoint.read_text())
            assert saved["protocol"] == protocol
            assert saved["seed"] == seed
            print(
                f"Phi seed {seed}: complete, skipping.",
                flush=True,
            )
            continue

        print(f"\nStarting Phi seed {seed}", flush=True)

        selection, evaluation = split_outputs(seed)

        # Entropy ranking uses only selection baseline outputs.
        entropies = []

        for prompt in prompts:
            frequencies = np.array(
                list(selection["unchanged"][prompt].values()),
                dtype=float,
            )
            probabilities = frequencies / frequencies.sum()

            entropies.append(float(
                -(probabilities * np.log2(probabilities)).sum()
            ))

        entropy_group = tuple(sorted(
            sorted(
                range(len(prompts)),
                key=lambda i: (-entropies[i], i),
            )[:n_selected]
        ))

        summary_rows = []
        subset_rows = []
        selections = {}

        for fold_index, (held_out, split) in enumerate(splits.items()):
            train_rng = np.random.default_rng(
                np.random.SeedSequence([seed, fold_index, 1])
            )
            test_rng = np.random.default_rng(
                np.random.SeedSequence([seed, fold_index, 2])
            )

            # Rank prompts without using the held-out type.
            aucs_by_type = {}

            for variant in split["selection"]:
                kind = phi_collections[variant]["job"]["kind"]
                assert kind != held_out

                null, changed = simulate_scores(
                    selection, variant, train_rng
                )

                prompt_aucs = np.array([
                    calculate_auc(null[i], changed[i])
                    for i in range(len(prompts))
                ])

                aucs_by_type.setdefault(kind, []).append(
                    prompt_aucs
                )

            # Equal weighting across selection types.
            ranking_scores = np.mean([
                np.mean(values, axis=0)
                for values in aucs_by_type.values()
            ], axis=0)

            ranked = sorted(
                range(len(prompts)),
                key=lambda i: (-ranking_scores[i], i),
            )
            performance_group = tuple(sorted(
                ranked[:n_selected]
            ))

            selections[held_out] = {
                "performance_prompts": [
                    prompts[i] for i in performance_group
                ],
                "entropy_prompts": [
                    prompts[i] for i in entropy_group
                ],
                "ranking_scores": {
                    prompt: float(ranking_scores[i])
                    for i, prompt in enumerate(prompts)
                },
            }

            variant_group_aucs = []

            for variant in split["evaluation"]:
                assert (
                    phi_collections[variant]["job"]["kind"]
                    == held_out
                )

                null, changed = simulate_scores(
                    evaluation, variant, test_rng
                )

                # Same simulated observations for all strategies.
                group_null = null[group_indices].mean(axis=1)
                group_changed = changed[group_indices].mean(axis=1)

                variant_group_aucs.append(np.array([
                    calculate_auc(
                        group_null[i], group_changed[i]
                    )
                    for i in range(len(groups))
                ]))

            # Equal weighting across held-out variants.
            mean_aucs = np.mean(variant_group_aucs, axis=0)

            performance_auc = float(
                mean_aucs[groups.index(performance_group)]
            )
            entropy_auc = float(
                mean_aucs[groups.index(entropy_group)]
            )
            random_auc = float(mean_aucs.mean())

            summary_rows.append({
                "seed": seed,
                "held_out_type": held_out,
                "performance_auc": performance_auc,
                "entropy_auc": entropy_auc,
                "random_mean_auc": random_auc,
                "random_subset_sd": float(mean_aucs.std()),
                "gain_vs_entropy": performance_auc - entropy_auc,
                "gain_vs_random": performance_auc - random_auc,
            })

            for group, auc in zip(groups, mean_aucs):
                subset_rows.append({
                    "seed": seed,
                    "held_out_type": held_out,
                    "prompt_indices": list(group),
                    "prompts": [prompts[i] for i in group],
                    "auc": float(auc),
                })

            print(
                f"  {held_out}: "
                f"performance={performance_auc:.4f}, "
                f"entropy={entropy_auc:.4f}, "
                f"random={random_auc:.4f}",
                flush=True,
            )

        result = {
            "seed": seed,
            "protocol": protocol,
            "selections": selections,
            "summary": summary_rows,
            "subset_results": subset_rows,
        }

        temporary = checkpoint.with_suffix(".tmp")
        temporary.write_text(
            json.dumps(result, indent=2, ensure_ascii=False)
        )
        temporary.replace(checkpoint)

        print(f"Saved Phi seed {seed}.", flush=True)

    # Reload checkpoints to assemble the complete result table.
    rows = []

    for seed in seeds:
        saved = json.loads(
            (PHI_REPEAT_DIR / f"seed_{seed}.json").read_text()
        )
        assert saved["protocol"] == protocol
        rows.extend(saved["summary"])

    results = pd.DataFrame(rows)

    assert len(results) == 15
    assert not results.duplicated(
        ["seed", "held_out_type"]
    ).any()

    results.to_csv(
        PHI_REPEAT_DIR / "all_seeds_comparison.csv",
        index=False,
    )

    return results


phi_repeated_results = run_phi_extension()

print("\nAll Phi repetitions completed.")
display(phi_repeated_results.round(4))
print("\nSaved in:", PHI_REPEAT_DIR)

### Original cell 49
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import pandas as pd

phi_results = pd.read_csv(
    PHI_REPEAT_DIR / "all_seeds_comparison.csv"
)

assert len(phi_results) == 15
assert not phi_results.duplicated(
    ["seed", "held_out_type"]
).any()

rows = []

for kind, group in phi_results.groupby("held_out_type"):
    assert set(group["seed"]) == {101, 202, 303, 404, 505}

    row = {
        "held_out_type": kind,
        "n_seeds": len(group),
    }

    for metric in [
        "performance_auc",
        "entropy_auc",
        "random_mean_auc",
        "gain_vs_entropy",
        "gain_vs_random",
    ]:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_sd"] = group[metric].std(ddof=1)

    row["positive_vs_entropy"] = int(
        (group["gain_vs_entropy"] > 0).sum()
    )
    row["positive_vs_random"] = int(
        (group["gain_vs_random"] > 0).sum()
    )

    rows.append(row)

phi_metrics_summary = pd.DataFrame(rows)

phi_metrics_summary.to_csv(
    PHI_REPEAT_DIR / "metrics_summary_across_seeds.csv",
    index=False,
)

display_rows = []

for _, row in phi_metrics_summary.iterrows():
    display_rows.append({
        "Held-out type": row["held_out_type"],
        "Performance AUC": (
            f"{row['performance_auc_mean']:.4f} ± "
            f"{row['performance_auc_sd']:.4f}"
        ),
        "Entropy AUC": (
            f"{row['entropy_auc_mean']:.4f} ± "
            f"{row['entropy_auc_sd']:.4f}"
        ),
        "Random AUC": (
            f"{row['random_mean_auc_mean']:.4f} ± "
            f"{row['random_mean_auc_sd']:.4f}"
        ),
        "Mean gain vs entropy": row["gain_vs_entropy_mean"],
        "Mean gain vs random": row["gain_vs_random_mean"],
        "Positive vs entropy": f"{int(row['positive_vs_entropy'])}/5",
        "Positive vs random": f"{int(row['positive_vs_random'])}/5",
    })

display(pd.DataFrame(display_rows).round(4))

print("± indicates standard deviation across five seeds, not a confidence interval.")
print("All strategies use 30 detection requests per test.")
print("Saved:", PHI_REPEAT_DIR / "metrics_summary_across_seeds.csv")

### Original cell 50
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
import pandas as pd

selection_rows = []

for seed in [101, 202, 303, 404, 505]:
    saved = json.loads(
        (PHI_REPEAT_DIR / f"seed_{seed}.json").read_text()
    )

    for held_out, choices in saved["selections"].items():
        metrics = next(
            row for row in saved["summary"]
            if row["held_out_type"] == held_out
        )

        selection_rows.append({
            "seed": seed,
            "held_out_type": held_out,
            "performance_prompts": repr(
                choices["performance_prompts"]
            ),
            "entropy_prompts": repr(
                choices["entropy_prompts"]
            ),
            "performance_auc": metrics["performance_auc"],
            "gain_vs_random": metrics["gain_vs_random"],
        })

phi_selection_check = pd.DataFrame(selection_rows)

print("PROMPT SELECTIONS WHEN RANDOM PRUNING IS HELD OUT")

with pd.option_context("display.max_colwidth", 150):
    display(
        phi_selection_check[
            phi_selection_check["held_out_type"] == "random"
        ].sort_values("seed").round(4)
    )

phi_selection_check.to_csv(
    PHI_REPEAT_DIR / "selected_prompts_across_seeds.csv",
    index=False,
)

print(
    "Saved:",
    PHI_REPEAT_DIR / "selected_prompts_across_seeds.csv",
)

### Original cell 51
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import subprocess
import shutil

PHI_ROOT = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
)
PHI_LARGE = PHI_ROOT / "expanded_scales_1000samples_v1"

print("GPU CHECK")
nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi is None:
    print(
        "nvidia-smi not found. Select:\n"
        "Runtime → Change runtime type → T4 GPU"
    )
else:
    result = subprocess.run(
        [nvidia_smi],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("Exit code:", result.returncode)

print("\nSAVED PYTHON SCRIPTS")
scripts = sorted(PHI_ROOT.rglob("*.py"))

for path in scripts:
    print(
        path.relative_to(PHI_ROOT),
        "| bytes:", path.stat().st_size,
    )

if not scripts:
    print("No saved Python scripts found.")

print("\nMANIFEST SETTINGS")
manifest = json.loads(
    (PHI_LARGE / "manifest.json").read_text()
)

# Print only known experiment settings, avoiding credentials.
config = manifest["config"]

for key in [
    "model", "n_samples", "temperature", "batch_size",
    "input_fraction", "seed", "max_model_len",
    "max_num_seqs", "max_num_batched_tokens",
]:
    if key in config:
        print(f"{key}: {config[key]}")

print("Number of prompts:", len(config["prompts"]))
print("Number of jobs:", len(manifest["jobs"]))

### Original cell 52
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import json

PHI_ROOT = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
)
PHI_LARGE = PHI_ROOT / "expanded_scales_1000samples_v1"

worker_path = PHI_LARGE / "worker.py"
assert worker_path.exists(), worker_path

print("SAVED WORKER")
for number, line in enumerate(
    worker_path.read_text().splitlines(), start=1
):
    print(f"{number:3}: {line}")

print("\nPOSSIBLE ENVIRONMENT / LAUNCHER RECORDS")
keywords = (
    "environment", "version", "requirements",
    "launch", "setup", "lock",
)

for path in sorted(PHI_ROOT.rglob("*")):
    if (
        path.is_file()
        and any(word in path.name.lower() for word in keywords)
        and path.suffix.lower() in {
            ".json", ".txt", ".toml", ".lock", ".py", ".log"
        }
    ):
        print(path.relative_to(PHI_ROOT))

print("\nSAVED BASELINE METADATA KEYS")
baseline = json.loads(
    (PHI_LARGE / "unchanged.json").read_text()
)
print(list(baseline))

print("\nCONFIGURATION FIELD NAMES")
print(list(baseline.get("config", {})))

### Original cell 53
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tomllib
import json

PHI_REPO = Path("/content/b3it_phi4mini_repo")
PHI_PROJECT = PHI_REPO / "track-llm-apis"

SOURCE_COMMIT = "ad2f69295797a5f03e7b480246080d91087d9007"
REPO_URL = (
    "https://github.com/timothee-chauvin/"
    "token-efficient-change-detection-llm-apis.git"
)

def git_command(arguments):
    result = subprocess.run(
        ["git", *arguments],
        capture_output=True,
        text=True,
    )
    if result.returncode:
        raise RuntimeError(
            result.stderr.strip() or result.stdout.strip()
        )
    return result.stdout.strip()

if not PHI_REPO.exists():
    print("Restoring repository...")
    git_command(["clone", REPO_URL, str(PHI_REPO)])

# Avoid changing an existing modified checkout.
changes = git_command([
    "-C", str(PHI_REPO), "status", "--porcelain"
])

if changes:
    raise RuntimeError(
        "This repository has local modifications. "
        "Stopping to preserve them:\n" + changes
    )

git_command([
    "-C", str(PHI_REPO),
    "checkout", "--detach", SOURCE_COMMIT,
])

print("Repository commit:")
print(git_command([
    "-C", str(PHI_REPO), "rev-parse", "HEAD"
]))

print("\nCOLAB PYTHON")
print(sys.version)

python_version_file = PHI_PROJECT / ".python-version"
if python_version_file.exists():
    print("\nPROJECT PYTHON")
    print(python_version_file.read_text().strip())

project = tomllib.loads(
    (PHI_PROJECT / "pyproject.toml").read_text()
)

print("\nREQUIRED PYTHON")
print(project.get("project", {}).get("requires-python"))

print("\nPROJECT DEPENDENCIES")
for dependency in project.get("project", {}).get("dependencies", []):
    print(dependency)

lock_path = PHI_PROJECT / "uv.lock"

if lock_path.exists():
    lock = tomllib.loads(lock_path.read_text())

    print("\nLOCKED KEY PACKAGE VERSIONS")
    wanted = {
        "torch", "transformers", "vllm",
        "datasets", "numpy", "tokenizers",
    }

    for package in lock.get("package", []):
        if package.get("name") in wanted:
            print(
                package["name"],
                package.get("version", "local"),
            )

print("\nSAVED INFERENCE SETTINGS")
manifest = json.loads(
    (PHI_LARGE / "manifest.json").read_text()
)
print(json.dumps(
    manifest["config"]["inference_settings"],
    indent=2,
))

### Original cell 54
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

PHI_PROJECT = Path(
    "/content/b3it_phi4mini_repo/track-llm-apis"
)
PHI_LARGE = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)

assert (PHI_PROJECT / "uv.lock").exists()
assert PHI_LARGE.exists()

# Install uv in Colab's environment.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "uv==0.13.1"],
    check=True,
)

install_env = os.environ.copy()
install_env["MPLBACKEND"] = "Agg"
install_env["UV_NO_PROGRESS"] = "1"

log_path = PHI_LARGE / "environment_restore.log"

command = [
    sys.executable, "-m", "uv",
    "sync",
    "--frozen",
    "--python", "3.12",
]

print("Installing locked project dependencies.", flush=True)
print("Log:", log_path, flush=True)

with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=PHI_PROJECT,
        env=install_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    returncode = process.wait()

if returncode != 0:
    raise RuntimeError(
        f"Installation failed ({returncode}). "
        f"Send the final error lines from {log_path}."
    )

PHI_PYTHON = PHI_PROJECT / ".venv/bin/python"
assert PHI_PYTHON.exists()

print("\nEnvironment restored.")
print("Project Python:", PHI_PYTHON)

### Original cell 55
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import os
import subprocess

PHI_PROJECT = Path(
    "/content/b3it_phi4mini_repo/track-llm-apis"
)
PHI_PYTHON = PHI_PROJECT / ".venv/bin/python"

PHI_LARGE = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)

check_code = r'''
import json
import platform
import tempfile
from pathlib import Path
from importlib.metadata import version

import torch
from transformers import AutoTokenizer

folder = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)
cfg = json.loads((folder / "manifest.json").read_text())["config"]

print("Python:", platform.python_version())

versions = {
    package: version(package)
    for package in [
        "torch", "transformers", "vllm",
        "tokenizers", "datasets", "numpy",
    ]
}
for package, package_version in versions.items():
    print(package + ":", package_version)

assert torch.cuda.is_available(), "CUDA is unavailable."

print("GPU:", torch.cuda.get_device_name(0))

original = AutoTokenizer.from_pretrained(cfg["model"])

# Reproduce the worker's tokenizer save/reload procedure.
with tempfile.TemporaryDirectory(prefix="phi_tokenizer_audit_") as tmp:
    original.save_pretrained(tmp)
    reloaded = AutoTokenizer.from_pretrained(tmp)

    probes = list(dict.fromkeys(
        cfg["prompts"] + [
            "Hello, world!",
            " leading space",
            "two  spaces",
            "line one\nline two",
            "café naïve",
            "مرحبا",
            "你好",
            "12345 + 678",
            "```python\nprint('hello')\n```",
            "<|user|>\nHello<|end|>\n<|assistant|>",
        ]
    ))

    mismatches = []

    for text in probes:
        first = original.encode(text, add_special_tokens=False)
        second = reloaded.encode(text, add_special_tokens=False)

        if first != second:
            mismatches.append(repr(text))

    vocab_equal = original.get_vocab() == reloaded.get_vocab()

    special_ids_equal = (
        original.all_special_ids == reloaded.all_special_ids
        and original.all_special_tokens == reloaded.all_special_tokens
        and original.bos_token_id == reloaded.bos_token_id
        and original.eos_token_id == reloaded.eos_token_id
        and original.pad_token_id == reloaded.pad_token_id
    )

    print("\nTokenizer class:", type(original).__name__)
    print("Vocabulary equal:", vocab_equal)
    print("Special tokens and IDs equal:", special_ids_equal)
    print("Texts checked:", len(probes))
    print("Tokenization mismatches:", mismatches)

    report = {
        "python": platform.python_version(),
        "versions": versions,
        "gpu": torch.cuda.get_device_name(0),
        "model": cfg["model"],
        "vocabulary_equal": vocab_equal,
        "special_tokens_equal": special_ids_equal,
        "probe_count": len(probes),
        "mismatches": mismatches,
        "scope": (
            "Original tokenizer versus tokenizer saved and reloaded "
            "in the restored environment. Not an audit of previous "
            "deleted checkpoints or every possible input."
        ),
    }

    # Separate audit file: existing collections remain unchanged.
    report_path = folder / "recovery_tokenizer_audit_v1.json"
    report_path.write_text(json.dumps(report, indent=2))

    print("Audit saved:", report_path)

    assert vocab_equal, "Tokenizer vocabulary changed after saving."
    assert special_ids_equal, "Special token configuration changed."
    assert not mismatches, "Tokenization differs for tested inputs."

print("\nEnvironment and tokenizer checks passed.")
'''

check_env = os.environ.copy()
check_env["MPLBACKEND"] = "Agg"

subprocess.run(
    [str(PHI_PYTHON), "-u", "-c", check_code],
    cwd=PHI_PROJECT,
    env=check_env,
    check=True,
)

### Original cell 56
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import ast
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone

PHI_ROOT = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
)
PHI_LARGE = PHI_ROOT / "expanded_scales_1000samples_v1"
PHI_PROJECT = Path(
    "/content/b3it_phi4mini_repo/track-llm-apis"
)
PHI_PYTHON = PHI_PROJECT / ".venv/bin/python"

# Check the saved tokenizer audit.
audit = json.loads(
    (PHI_LARGE / "recovery_tokenizer_audit_v1.json").read_text()
)

print("Environment versions:", audit["versions"])
print("GPU:", audit["gpu"])
print("Vocabulary equal:", audit["vocabulary_equal"])
print("Special tokens equal:", audit["special_tokens_equal"])
print("Tokenization mismatches:", audit["mismatches"])

assert audit["vocabulary_equal"]
assert audit["special_tokens_equal"]
assert not audit["mismatches"]
assert PHI_PYTHON.exists()

# Restore the exact saved worker locally.
PHI_WORK = Path("/content/b3it_phi4mini_recovery")
PHI_WORK.mkdir(parents=True, exist_ok=True)

worker = PHI_WORK / "worker.py"
worker_source = (PHI_LARGE / "worker.py").read_text()
ast.parse(worker_source)
worker.write_text(worker_source)

manifest = json.loads(
    (PHI_LARGE / "manifest.json").read_text()
)
cfg = manifest["config"]
baseline_jobs = [
    job for job in manifest["jobs"]
    if job["kind"] == "unchanged"
]
assert len(baseline_jobs) == 1
job = baseline_jobs[0]

# Keep recovery checks separate from the original collection.
CHECK_DIR = PHI_LARGE / "recovery_checks_v1"
CHECK_DIR.mkdir(parents=True, exist_ok=True)
output = CHECK_DIR / "unchanged_restored.json"

config_path = PHI_WORK / "config.json"
job_path = PHI_WORK / "unchanged_job.json"
config_path.write_text(json.dumps(cfg))
job_path.write_text(json.dumps(job))

def valid_baseline(path):
    if not path.exists():
        return False
    try:
        saved = json.loads(path.read_text())
        return (
            saved["config"] == cfg
            and saved["job"] == job
            and set(saved["counts"]) == set(cfg["prompts"])
            and all(
                all(type(v) is int and v >= 0 for v in counts.values())
                and sum(counts.values()) == cfg["n_samples"]
                for counts in saved["counts"].values()
            )
        )
    except (OSError, ValueError, KeyError, TypeError):
        return False

if output.exists():
    assert valid_baseline(output), (
        "A recovery file exists but failed validation. "
        "Stopping without overwriting it."
    )
    print("\nRecovery baseline already complete; skipping.")
else:
    run_env = os.environ.copy()
    run_env.update({
        "MPLBACKEND": "Agg",
        "PYTHONUNBUFFERED": "1",
        "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
        "VLLM_ALLOW_INSECURE_SERIALIZATION": "1",
    })
    run_env["PATH"] = (
        str(PHI_PYTHON.parent)
        + os.pathsep
        + run_env.get("PATH", "")
    )

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    log_path = CHECK_DIR / f"baseline_{stamp}.log"

    command = [
        str(PHI_PYTHON), "-u", str(worker), "sample",
        "--config", str(config_path),
        "--job", str(job_path),
        "--checkpoint", str(PHI_WORK / "unused_checkpoint"),
        "--output", str(output),
    ]

    print("\nCollecting recovery baseline.")
    print("Log:", log_path)

    with log_path.open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=PHI_PROJECT,
            env=run_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )

        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()

            returncode = process.wait()

        except BaseException:
            import signal
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            except ProcessLookupError:
                pass
            raise

    if returncode != 0:
        raise RuntimeError(
            f"Baseline collection failed ({returncode}). "
            f"See {log_path}"
        )

    assert valid_baseline(output), "Saved baseline failed validation."

print("\nRecovery baseline ready:", output)
print("Original unchanged.json preserved.")

### Original cell 57
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PHI_LARGE = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)
CHECK_DIR = PHI_LARGE / "recovery_checks_v1"

old = json.loads(
    (PHI_LARGE / "unchanged.json").read_text()
)
restored = json.loads(
    (CHECK_DIR / "unchanged_restored.json").read_text()
)

assert old["config"] == restored["config"]
assert old["job"] == restored["job"]
assert set(old["counts"]) == set(restored["counts"])

N_PERMUTATIONS = 5000
rng = np.random.default_rng(20261112)

rows = []
null_tv_by_prompt = []

for prompt in old["config"]["prompts"]:
    first = old["counts"][prompt]
    second = restored["counts"][prompt]

    tokens = sorted(set(first) | set(second))

    a = np.array(
        [first.get(token, 0) for token in tokens],
        dtype=np.int64,
    )
    b = np.array(
        [second.get(token, 0) for token in tokens],
        dtype=np.int64,
    )

    n_a, n_b = int(a.sum()), int(b.sum())
    assert n_a == n_b == 1000

    observed_tv = 0.5 * np.abs(
        a / n_a - b / n_b
    ).sum()

    # Randomly reassign pooled observations to two collections,
    # preserving each collection's original size.
    pooled = a + b

    permuted_a = rng.multivariate_hypergeometric(
        pooled, n_a, size=N_PERMUTATIONS
    )
    permuted_b = pooled - permuted_a

    null_tv = 0.5 * np.abs(
        permuted_a / n_a - permuted_b / n_b
    ).sum(axis=1)

    null_tv_by_prompt.append(null_tv)

    rows.append({
        "prompt": repr(prompt),
        "observed_tv": float(observed_tv),
        "null_mean_tv": float(null_tv.mean()),
    })

details = pd.DataFrame(rows)

observed_mean = float(details["observed_tv"].mean())
null_means = np.mean(null_tv_by_prompt, axis=0)

p_value = float(
    (1 + np.count_nonzero(null_means >= observed_mean))
    / (N_PERMUTATIONS + 1)
)

report = {
    "observed_mean_tv": observed_mean,
    "null_mean_tv": float(null_means.mean()),
    "null_95th_percentile": float(
        np.quantile(null_means, 0.95)
    ),
    "permutation_p_value": p_value,
    "n_permutations": N_PERMUTATIONS,
    "interpretation": (
        "Upper-tail check for a distribution difference, "
        "assuming outputs are exchangeable within each prompt. "
        "A non-small p-value does not prove equivalence."
    ),
}

display(details.round(4))
display(pd.DataFrame([{
    key: value
    for key, value in report.items()
    if key != "interpretation"
}]).round(5))

details.to_csv(
    CHECK_DIR / "baseline_tv_per_prompt.csv",
    index=False,
)
(CHECK_DIR / "baseline_comparison.json").write_text(
    json.dumps(report, indent=2)
)

print("Saved baseline comparison:", CHECK_DIR)

### Original cell 58
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import json
import os
import signal
import subprocess
import tempfile
import time
import shutil

PHI_PROJECT = Path(
    "/content/b3it_phi4mini_repo/track-llm-apis"
)
PHI_PYTHON = PHI_PROJECT / ".venv/bin/python"
PHI_LARGE = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)
WORK = Path("/content/b3it_phi4mini_recovery")
WORK.mkdir(parents=True, exist_ok=True)

assert PHI_PYTHON.exists(), "Restore the environment first."

manifest = json.loads(
    (PHI_LARGE / "manifest.json").read_text()
)
cfg = manifest["config"]
jobs = manifest["jobs"]

assert cfg["n_samples"] == 1000

# Reuse the saved worker unchanged.
worker_source = (PHI_LARGE / "worker.py").read_text()
ast.parse(worker_source)

worker = WORK / "worker.py"
worker.write_text(worker_source)

config_path = WORK / "config.json"
config_path.write_text(json.dumps(cfg))

run_env = os.environ.copy()
run_env.update({
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "VLLM_ALLOW_INSECURE_SERIALIZATION": "1",
})
run_env["PATH"] = (
    str(PHI_PYTHON.parent)
    + os.pathsep
    + run_env.get("PATH", "")
)

LOGS = PHI_LARGE / "recovery_collection_logs"
LOGS.mkdir(parents=True, exist_ok=True)


def valid_result(path, job):
    if not path.exists():
        return False

    try:
        saved = json.loads(path.read_text())
        counts = saved["counts"]

        return (
            saved["config"] == cfg
            and saved["job"] == job
            and set(counts) == set(cfg["prompts"])
            and all(
                isinstance(outputs, dict)
                and all(
                    type(value) is int and value >= 0
                    for value in outputs.values()
                )
                and sum(outputs.values()) == cfg["n_samples"]
                for outputs in counts.values()
            )
        )
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return False


def run_action(action, job_path, checkpoint, output, job_name):
    stamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%S%fZ"
    )
    log_path = LOGS / f"{job_name}_{action}_{stamp}.log"

    command = [
        str(PHI_PYTHON), "-u", str(worker), action,
        "--config", str(config_path),
        "--job", str(job_path),
        "--checkpoint", str(checkpoint),
        "--output", str(output),
    ]

    print(f"\nStarting {action}: {job_name}", flush=True)
    print("Log:", log_path, flush=True)

    with log_path.open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=PHI_PROJECT,
            env=run_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )

        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()

            returncode = process.wait()

        except BaseException:
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            except ProcessLookupError:
                pass
            raise

    if returncode != 0:
        raise RuntimeError(
            f"{action} failed for {job_name}. See {log_path}"
        )


# Check existing files before starting expensive work.
for job in jobs:
    output = PHI_LARGE / f"{job['name']}.json"
    if output.exists() and not valid_result(output, job):
        raise RuntimeError(
            f"Existing result failed validation: {output}\n"
            "Stopping without overwriting it."
        )

for index, job in enumerate(jobs, start=1):
    name = job["name"]
    output = PHI_LARGE / f"{name}.json"

    print(f"\n[{index}/{len(jobs)}] {name}", flush=True)

    if valid_result(output, job):
        print("Already complete; skipping.", flush=True)
        continue

    job_path = WORK / "current_job.json"
    job_path.write_text(json.dumps(job))
    started = time.monotonic()

    # Only this cell's temporary checkpoint is removed afterward.
    with tempfile.TemporaryDirectory(
        prefix="variant_", dir=WORK
    ) as temporary:
        checkpoint = Path(temporary) / "checkpoint"

        if job["kind"] != "unchanged":
            run_action(
                "edit", job_path, checkpoint, output, name
            )

        # Editing has exited before inference starts.
        run_action(
            "sample", job_path, checkpoint, output, name
        )

        assert valid_result(output, job), (
            f"Saved output failed validation: {output}"
        )

    print(
        f"\nSaved and verified: {name} | "
        f"{(time.monotonic() - started) / 60:.1f} minutes",
        flush=True,
    )

completed = sum(
    valid_result(PHI_LARGE / f"{job['name']}.json", job)
    for job in jobs
)

print(f"\nCOMPLETED: {completed}/{len(jobs)}")
assert completed == len(jobs)
print("Verified: 18 versions × 10 prompts × 1,000 outputs.")

### Original cell 59
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import copy
import json

PHI_ROOT = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning"
)
PHI_1000_COLLECTION = PHI_ROOT / "expanded_scales_1000samples_v1"

manifest = json.loads(
    (PHI_1000_COLLECTION / "manifest.json").read_text()
)
cfg = manifest["config"]
PHI_1000_DATA = {}

for job in manifest["jobs"]:
    path = PHI_1000_COLLECTION / f"{job['name']}.json"
    saved = json.loads(path.read_text())

    assert saved["config"] == cfg, path.name
    assert saved["job"] == job, path.name
    assert set(saved["counts"]) == set(cfg["prompts"]), path.name

    for counts in saved["counts"].values():
        assert all(
            type(value) is int and value >= 0
            for value in counts.values()
        ), path.name
        assert sum(counts.values()) == 1000, path.name

    PHI_1000_DATA[job["name"]] = saved

assert len(PHI_1000_DATA) == 18

# Preserve the previous evaluation choices.
old_protocol_path = (
    PHI_ROOT / "expanded_scales_100samples_v1"
    / "extension_selection_repeated_v1" / "protocol.json"
)
old_protocol = json.loads(old_protocol_path.read_text())

PHI_1000_PROTOCOL = copy.deepcopy(old_protocol)

assert PHI_1000_PROTOCOL["candidate_prompts"] == cfg["prompts"]

PHI_1000_PROTOCOL["selection_pool_size"] = 500
PHI_1000_PROTOCOL["evaluation_pool_size"] = 500
PHI_1000_PROTOCOL["limitations"] = [
    "Ten previously collected candidate Border Inputs",
    "Repeated splits reuse the same recorded collection",
    "Entropy baseline is not a verified original-repository rule",
    "Collection spans sessions; recovery checks are recorded separately",
    "Tokenizer regex warning remains an unresolved audit point",
]

PHI_1000_DIR = (
    PHI_1000_COLLECTION / "extension_selection_repeated_v1"
)
PHI_1000_DIR.mkdir(parents=True, exist_ok=True)

PHI_1000_PROTOCOL_PATH = PHI_1000_DIR / "protocol.json"

if PHI_1000_PROTOCOL_PATH.exists():
    assert (
        json.loads(PHI_1000_PROTOCOL_PATH.read_text())
        == PHI_1000_PROTOCOL
    ), "A different protocol already exists. Do not overwrite it."
    print("Matching protocol already exists.")
else:
    PHI_1000_PROTOCOL_PATH.write_text(
        json.dumps(PHI_1000_PROTOCOL, indent=2, ensure_ascii=False)
    )
    print("New protocol saved.")

print("Verified: 18 versions × 10 prompts × 1,000 outputs.")
print("Selection/evaluation pools: 500 / 500.")
print("Seeds:", PHI_1000_PROTOCOL["seeds"])
print("Detection requests per test: 30.")
print("Previous 100-output analysis preserved.")
print("Saved:", PHI_1000_PROTOCOL_PATH)

### Original cell 60
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
assert callable(globals().get("run_phi_extension")), (
    "The analysis function is missing after a runtime restart. "
    "Restore the function definition from Phi Cell 5 first, "
    "without running its final call."
)

# Temporarily connect the existing function to the larger dataset.
bindings = {
    "phi_protocol_path": PHI_1000_PROTOCOL_PATH,
    "PHI_PROTOCOL": PHI_1000_PROTOCOL,
    "phi_collections": PHI_1000_DATA,
    "PHI_REPEAT_DIR": PHI_1000_DIR,
}

missing = object()
previous = {
    name: globals().get(name, missing)
    for name in bindings
}

try:
    globals().update(bindings)

    phi_1000_results = run_phi_extension()

finally:
    # Restore previous variables even if the calculation is interrupted.
    for name, value in previous.items():
        if value is missing:
            globals().pop(name, None)
        else:
            globals()[name] = value

print("\nPHI: 1,000-OUTPUT EXTENSION RESULTS")
display(phi_1000_results.round(4))
print("\nSaved:", PHI_1000_DIR)

### Original cell 61
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd

PHI_1000_DIR = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1/"
    "extension_selection_repeated_v1"
)

results = pd.read_csv(
    PHI_1000_DIR / "all_seeds_comparison.csv"
)

assert len(results) == 15
assert not results.duplicated(["seed", "held_out_type"]).any()

summary_rows = []

for kind, group in results.groupby("held_out_type"):
    assert set(group["seed"]) == {101, 202, 303, 404, 505}

    row = {"held_out_type": kind, "n_seeds": len(group)}

    for metric in [
        "performance_auc",
        "entropy_auc",
        "random_mean_auc",
        "gain_vs_entropy",
        "gain_vs_random",
    ]:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_sd"] = group[metric].std(ddof=1)

    row["positive_vs_entropy"] = int(
        (group["gain_vs_entropy"] > 0).sum()
    )
    row["positive_vs_random"] = int(
        (group["gain_vs_random"] > 0).sum()
    )

    summary_rows.append(row)

phi_1000_summary = pd.DataFrame(summary_rows)

phi_1000_summary.to_csv(
    PHI_1000_DIR / "metrics_summary_across_seeds.csv",
    index=False,
)

display_rows = []

for _, row in phi_1000_summary.iterrows():
    display_rows.append({
        "Held-out type": row["held_out_type"],
        "Performance AUC": (
            f"{row['performance_auc_mean']:.4f} ± "
            f"{row['performance_auc_sd']:.4f}"
        ),
        "Entropy AUC": (
            f"{row['entropy_auc_mean']:.4f} ± "
            f"{row['entropy_auc_sd']:.4f}"
        ),
        "Random AUC": (
            f"{row['random_mean_auc_mean']:.4f} ± "
            f"{row['random_mean_auc_sd']:.4f}"
        ),
        "Mean gain vs entropy": row["gain_vs_entropy_mean"],
        "Mean gain vs random": row["gain_vs_random_mean"],
        "Positive vs entropy": f"{int(row['positive_vs_entropy'])}/5",
        "Positive vs random": f"{int(row['positive_vs_random'])}/5",
    })

display(pd.DataFrame(display_rows).round(4))

print("± indicates SD across seeds, not a confidence interval.")
print("Recorded outputs per prompt/version: 1,000.")
print("Selection/evaluation pools: 500 / 500.")
print("Detection budget: 30 requests per test.")
print("Summary saved:", PHI_1000_DIR)

### Original cell 62
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive")

directories = {
    "Qwen2.5-0.5B": (
        BASE / "B3IT_Qwen05_Noise_Pruning"
        / "expanded_scales_1000samples_v1"
        / "extension_selection_repeated_v1"
    ),
    "Phi-4-mini": (
        BASE / "B3IT_Phi4mini_Noise_Pruning"
        / "expanded_scales_1000samples_v1"
        / "extension_selection_repeated_v1"
    ),
}

types = ["magnitude", "random", "random_noise"]
type_labels = [
    "Magnitude\npruning",
    "Random\npruning",
    "Gaussian\nnoise",
]

strategies = [
    ("performance_auc", "Performance-based", "#1565C0"),
    ("entropy_auc", "Entropy-based", "#EF6C00"),
    ("random_mean_auc", "Uniform random (mean)", "#607D8B"),
]

fig, axes = plt.subplots(
    1, 2, figsize=(11, 5), sharey=True
)

for ax, (model, directory) in zip(axes, directories.items()):
    results = pd.read_csv(directory / "all_seeds_comparison.csv")

    assert len(results) == 15
    assert not results.duplicated(
        ["seed", "held_out_type"]
    ).any()
    assert set(results["held_out_type"]) == set(types)

    for kind in types:
        assert set(
            results.loc[
                results["held_out_type"] == kind, "seed"
            ]
        ) == {101, 202, 303, 404, 505}

    x = np.arange(len(types))

    for offset, (metric, label, color) in zip(
        [-0.18, 0, 0.18], strategies
    ):
        stats = (
            results.groupby("held_out_type")[metric]
            .agg(["mean", "std"])
            .reindex(types)
        )

        ax.errorbar(
            x + offset,
            stats["mean"],
            yerr=stats["std"],
            fmt="o",
            markersize=6,
            capsize=4,
            color=color,
            label=label,
        )

    ax.set_title(model)
    ax.set_xticks(x, type_labels)
    ax.set_ylim(0.5, 1.02)
    ax.set_xlabel("Held-out perturbation type")
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

axes[0].set_ylabel("Mean held-out ROC AUC")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.91),
    ncol=3,
    frameon=False,
)

fig.suptitle(
    "Border Input selection at a fixed detection budget",
    fontsize=14,
    y=0.99,
)

fig.text(
    0.5, 0.02,
    "3 prompts × 10 detection samples; 50 reference samples per prompt.\n"
    "1,000 recorded outputs split 500/500. Points: five-seed means; "
    "bars: ±1 SD, not confidence intervals.\n"
    "Random baseline averages 120 subsets. "
    "Discovery, selection and reference-collection costs excluded.",
    ha="center",
    fontsize=9,
)

fig.tight_layout(rect=[0, 0.18, 1, 0.84])

# Save beside the new Phi analysis.
output = directories["Phi-4-mini"] / "qwen_phi_selection_comparison"

fig.savefig(
    output.with_suffix(".png"), dpi=300, bbox_inches="tight"
)
fig.savefig(
    output.with_suffix(".pdf"), bbox_inches="tight"
)

plt.show()

print("PNG:", output.with_suffix(".png"))
print("PDF:", output.with_suffix(".pdf"))

### Original cell 63
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning")
folder = root / "expanded_scales_1000samples_v1"
python = Path(
    "/content/b3it_phi4mini_repo/track-llm-apis/.venv/bin/python"
)

print("Google Drive accessible :", root.exists())
print("Manifest disponible :", (folder / "manifest.json").exists())
print("Environnement disponible :", python.exists())
print(
    "Audit précédent disponible :",
    (folder / "recovery_tokenizer_audit_v1.json").exists(),
)

### Original cell 64
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import json
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)

audit = json.loads(
    (folder / "recovery_tokenizer_audit_v1.json").read_text()
)

print("AUDIT SAUVEGARDÉ")
print(json.dumps(audit, indent=2, ensure_ascii=False))

print("\nDÉPÔT LOCAL")
repo = Path("/content/b3it_phi4mini_repo/track-llm-apis")
print("Dossier présent :", repo.exists())
print("Fichier uv.lock présent :", (repo / "uv.lock").exists())

### Original cell 65
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/b3it_phi4mini_repo")
commit = "ad2f69295797a5f03e7b480246080d91087d9007"

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/timothee-chauvin/"
            "token-efficient-change-detection-llm-apis.git",
            str(repo),
        ],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(repo), "checkout", "--detach", commit],
    check=True,
)

project = repo / "track-llm-apis"

assert (project / "uv.lock").exists(), "Fichier uv.lock introuvable."

print("Dépôt restauré :", project)
print(
    "Commit :",
    subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"],
        text=True,
    ).strip(),
)

print("\nUV disponible :")
subprocess.run(["python", "-m", "uv", "--version"], check=False)

### Original cell 66
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

project = Path("/content/b3it_phi4mini_repo/track-llm-apis")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "uv==0.13.1"],
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "uv",
        "sync", "--frozen", "--python", "3.12",
    ],
    cwd=project,
    check=True,
)

python = project / ".venv/bin/python"

env = os.environ.copy()
env["MPLBACKEND"] = "agg"

subprocess.run(
    [
        str(python), "-c",
        """
import sys
from importlib.metadata import version

print("Python :", sys.version.split()[0])
for package in [
    "torch", "transformers", "vllm",
    "tokenizers", "datasets", "numpy"
]:
    print(f"{package} : {version(package)}")
""",
    ],
    env=env,
    check=True,
)

print("\nEnvironnement restauré.")

### Original cell 67
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import subprocess
from pathlib import Path

project = Path("/content/b3it_phi4mini_repo/track-llm-apis")
python = project / ".venv/bin/python"

code = r'''
from importlib.metadata import version
from pathlib import Path
import transformers

print("VERSIONS")
for name in ["transformers", "tokenizers", "vllm"]:
    print(name, version(name))

root = Path(transformers.__file__).parent
phrase = "incorrect regex pattern"

print("\nCODE À L’ORIGINE DE L’AVERTISSEMENT")
found = False

for path in root.rglob("*.py"):
    text = path.read_text(errors="replace")
    if phrase not in text:
        continue

    found = True
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if phrase in line:
            print("\nFichier :", path.relative_to(root))
            for j in range(max(0, i - 25), min(len(lines), i + 20)):
                print(f"{j + 1}: {lines[j]}")

if not found:
    print("Texte de l’avertissement introuvable.")
'''

result = subprocess.run(
    [str(python), "-c", code],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode:
    print(result.stderr)

### Original cell 68
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import os
import subprocess

code = r'''
import json
import tempfile
from datetime import datetime, timezone
from pathlib import Path

from transformers import AutoConfig, AutoTokenizer
from track_llm_apis.util import load_lmsys_chat_1m

folder = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)
cfg = json.loads((folder / "manifest.json").read_text())["config"]

dataset = load_lmsys_chat_1m(
    use_cache=True,
    datasets_dir=Path(cfg["datasets_dir"]),
)
traffic = [row["conversation"][0]["content"] for row in dataset]
texts = list(cfg["prompts"]) + traffic

original = AutoTokenizer.from_pretrained(cfg["model"])

with tempfile.TemporaryDirectory() as tmp:
    original.save_pretrained(tmp)
    AutoConfig.from_pretrained(cfg["model"]).save_pretrained(tmp)
    local = AutoTokenizer.from_pretrained(tmp)

    checks = {}
    for special in [False, True]:
        differences = []
        for i, text in enumerate(texts):
            if original.encode(text, add_special_tokens=special) != local.encode(
                text, add_special_tokens=special
            ):
                differences.append(i)

        checks[str(special)] = {
            "mismatch_count": len(differences),
            "first_mismatch_indices": differences[:20],
        }

    report = {
        "model": cfg["model"],
        "border_inputs": len(cfg["prompts"]),
        "traffic_texts": len(traffic),
        "vocabulary_equal": original.get_vocab() == local.get_vocab(),
        "special_tokens_equal":
            original.special_tokens_map == local.special_tokens_map,
        "encoding_checks_by_add_special_tokens": checks,
        "scope": (
            "Original versus newly saved local tokenizer, with model config. "
            "Does not inspect deleted checkpoints or the vLLM loading path."
        ),
    }

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output = folder / f"recovery_tokenizer_full_text_audit_{stamp}.json"
output.write_text(json.dumps(report, indent=2, ensure_ascii=False))

print(json.dumps(report, indent=2, ensure_ascii=False))
print("\nSaved:", output)
'''

env = os.environ.copy()
env["MPLBACKEND"] = "agg"

result = subprocess.run(
    [str(python), "-c", code],
    cwd=project,
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode:
    print(result.stderr)

assert result.returncode == 0, "Vérification interrompue : voir l’erreur ci-dessus."

### Original cell 69
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import subprocess

code = r'''
import inspect
from vllm.transformers_utils.tokenizer import get_tokenizer

print("SIGNATURE")
print(inspect.signature(get_tokenizer))

print("\nCODE DU CHARGEMENT")
print(inspect.getsource(get_tokenizer))
'''

result = subprocess.run(
    [str(python), "-c", code],
    cwd=project,
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode:
    print(result.stderr)

### Original cell 70
Source: `B3IT_Phi4mini_Noise_Pruning (1).ipynb`.

In [ ]:
import subprocess

code = r'''
import json
import tempfile
from datetime import datetime, timezone
from pathlib import Path

from transformers import AutoConfig, AutoTokenizer
from vllm.transformers_utils.tokenizer import get_tokenizer
from track_llm_apis.util import load_lmsys_chat_1m

folder = Path(
    "/content/drive/MyDrive/B3IT_Phi4mini_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)
cfg = json.loads((folder / "manifest.json").read_text())["config"]

dataset = load_lmsys_chat_1m(
    use_cache=True,
    datasets_dir=Path(cfg["datasets_dir"]),
)
traffic = [row["conversation"][0]["content"] for row in dataset]
texts = list(cfg["prompts"]) + traffic

original = get_tokenizer(cfg["model"])

with tempfile.TemporaryDirectory() as tmp:
    AutoTokenizer.from_pretrained(cfg["model"]).save_pretrained(tmp)
    AutoConfig.from_pretrained(cfg["model"]).save_pretrained(tmp)
    local = get_tokenizer(tmp)

    checks = {}
    for special in [False, True]:
        mismatches = [
            i for i, text in enumerate(texts)
            if original.encode(text, add_special_tokens=special)
            != local.encode(text, add_special_tokens=special)
        ]
        checks[str(special)] = {
            "mismatch_count": len(mismatches),
            "first_mismatch_indices": mismatches[:20],
        }

    # Output strings are used as categories in the saved counts.
    original_vocab = original.get_vocab()
    local_vocab = local.get_vocab()
    ids = sorted(set(original_vocab.values()) | set(local_vocab.values()))

    decoding_mismatches = [
        token_id for token_id in ids
        if original.decode([token_id]) != local.decode([token_id])
    ]

    report = {
        "model": cfg["model"],
        "loader": "vLLM get_tokenizer, default settings",
        "border_inputs": len(cfg["prompts"]),
        "traffic_texts": len(traffic),
        "vocabulary_equal": original_vocab == local_vocab,
        "encoding_checks_by_add_special_tokens": checks,
        "single_token_decoding_mismatch_count": len(decoding_mismatches),
        "first_decoding_mismatch_ids": decoding_mismatches[:20],
        "scope": (
            "Original versus newly saved local tokenizer through vLLM. "
            "Previous deleted checkpoints cannot be inspected."
        ),
    }

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output = folder / f"recovery_vllm_tokenizer_audit_{stamp}.json"
output.write_text(json.dumps(report, indent=2, ensure_ascii=False))

print(json.dumps(report, indent=2, ensure_ascii=False))
print("\nSaved:", output)
'''

result = subprocess.run(
    [str(python), "-c", code],
    cwd=project,
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode:
    print(result.stderr)

assert result.returncode == 0, "Vérification interrompue : voir l’erreur."